# Final question set - manual answers

This notebook manually computes the answers the full task set so that the gold tool paths can be verified. Under each task template, we organise the questions in the following order:
- answerable questions on the Easy dataset (E1 and E2)
- answerable questions on the Hard dataset (H1 and H2, sometimes H3)
- abstention questions on the Hard dataset (H3)

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
pd.set_option("display.precision", 4)

In [2]:
df_questions = pd.read_csv("../benchmark_outputs/v2/question_set_v2_params.csv")
df_questions = df_questions[["task_id", "template", "dataset", "answerable", "question"]]
df_questions.head()

,task_id,template,dataset,answerable,question
0,T1-E1,T1- Single value retrieval,easy,yes,What proportion of Banking reviews that mentio...
1,T1-E2,T1- Single value retrieval,easy,yes,What proportion of Fashion reviews that mentio...
2,T1-H1,T1- Single value retrieval,hard,yes,What proportion of Fashion reviews that mentio...
3,T1-H2,T1- Single value retrieval,hard,yes,What proportion of Price Comparison reviews th...
4,T1-H3,T1- Single value retrieval,hard,no (abstain),What proportion of Ride Hailing reviews that m...


In [3]:
# # select questions
# tpl     = df_questions["task_id"].str.split("-").str[0]
# order   = tpl.str[1:].astype(int)
# is_ans  = df_questions["answerable"].eq("yes")
# is_easy = df_questions["dataset"].eq("easy")

# easy = is_ans & is_easy
# hard = is_ans & ~is_easy

# df_manual = (pd.concat([df_questions[easy].groupby(tpl[easy]).head(1),   # 10 easy answerable
#                         df_questions[hard].groupby(tpl[hard]).head(1),   # 10 hard answerable
#                         df_questions[~is_ans]])                          # 10 abstention
#                .assign(_o=lambda d: order[d.index])
#                .sort_values(["_o", "task_id"])
#                .drop(columns="_o")
#                .reset_index(drop=True))

# print(len(df_manual), "tasks")
# df_manual

In [4]:
df_easy = pd.read_csv("../benchmark_outputs/v2/easy/labels.csv")
df_easy.head()

,review_id,industry,org,parent_aspect,child_aspect,sentiment
0,R000001,Consulting,Castellan Partners,staff-support,attitude-of-staff,negative
1,R000002,Price Comparison,PricePilot,value,price-value-for-money,negative
2,R000003,Price Comparison,CompareHive,account-management,account-access,negative
3,R000004,Consulting,Castellan Partners,staff-support,attitude-of-staff,positive
4,R000005,Consulting,Merrow & Pike,value,discounts-promotions,positive


In [5]:
df_hard = pd.read_csv("../benchmark_outputs/v2/hard/labels.csv")
df_hard.head()

,review_id,industry,org,parent_aspect,child_aspect,sentiment
0,R000001,Fashion,Northerly,company-brand,general-satisfaction,positive
1,R000002,Trading,Quantly,purchase-booking-experience,ease-of-use,positive
2,R000003,Fashion,Northerly,staff-support,attitude-of-staff,negative
3,R000004,Price Comparison,CompareHive,online-experience,app-website,negative
4,R000005,Banking,Halden Savings,purchase-booking-experience,ease-of-use,negative


### Conventions

- **Topic** = `child_aspect` (the 12 leaf aspects). `parent_aspect` is never used as a topic.
- **Negative rate** = negative reviews / *all* reviews mentioning that slice, i.e. neutral reviews stay in the denominator.
- **Complaint rate** for an organisation = its negative reviews / all its reviews.
- **Minimum volume** = 30 mentions, applied wherever the question asks for it.
- **Abstention rule**: abstain when the slice the question names has no rows at all, or has
  fewer than 30 reviews, or cannot support the shape of the answer asked for
  (e.g. a "top 3" where fewer than 3 topics exist). The evidence is printed in each case, and
  every abstention answer opens with `No answer.` followed by the reason.
- **Non-actionable topics**: `competitor`, `general-satisfaction` and `reviews` describe how customers
  feel overall rather than something an organisation can fix. They are dropped from the prescriptive
  recommendation only - the T8 ranking, the T9 "fix first" pick and the T10 roadmap - and kept
  everywhere else, so the T9 gap decomposition and the T10 pain-point ranking still report them.
- **Topic wording**: answers name topics in plain English, the printed tables keep the raw labels:

  | label | in the answer | label | in the answer |
  | --- | --- | --- | --- |
  | `account-access` | account access | `email` | email support |
  | `app-website` | the app or website | `general-satisfaction` | general satisfaction |
  | `attitude-of-staff` | staff attitude | `phone` | phone support |
  | `competitor` | comparisons with competitors | `price-value-for-money` | price and value for money |
  | `discounts-promotions` | discounts and promotions | `reviews` | other reviews and forums |
  | `ease-of-use` | ease of use | `speed` | speed |

- **Answer wording**: every cell computes its own evidence from scratch and then prints the answer
  it supports. The phrasing follows the "high-quality answer" examples in the task-set design
  document and is kept consistent within a template, but it is written out per question rather than
  produced by a shared builder, so the wording can follow what the numbers actually show.
  Percentages are reported to one decimal place; p-values as `p < 0.001` or to three decimals.
- **Statistical testing** = two-sided two-proportion z-test with pooled variance, significance at alpha = 0.05.
- **Priority score** (T8, and the T10 roadmap) = 0.5 x minmax(negative rate) + 0.5 x minmax(volume),
  exactly as the T8 question defines it, computed over the actionable topics that clear the
  30-mention floor. Min-max scaling is relative to the pool it is computed on, so the floor and
  the non-actionable drop are applied before the normalisation.


In [6]:
MIN_N = 30   # minimum mentions: used by the templates that name it, and as the
             # data-sufficiency floor for deciding abstention

# Aspects that describe how customers feel overall rather than something the organisation
# can fix. Dropped from the prescriptive recommendation (T8-T10); kept in descriptive and
# diagnostic analysis (T1-T7).
NON_ACTIONABLE = ["competitor", "general-satisfaction", "reviews"]

def ask(task_id):
    """Print the question and hand back the dataset it must be answered on."""
    r = df_questions.loc[df_questions.task_id == task_id].iloc[0]
    print(f"{r.task_id}  |  dataset: {r.dataset}  |  answerable: {r.answerable}")
    print(r.question)
    print("-" * 100)
    return df_easy if r.dataset == "easy" else df_hard

## T1- Single value retrieval

In [7]:
df = ask("T1-E1")

sub = df[(df.industry == "Banking") & (df.child_aspect == "app-website")] # filter
n = len(sub) # count total
k = sub.sentiment.eq("negative").sum() # count negative
print(sub.sentiment.value_counts().to_string())
print(f"\nmentions = {n}, negative = {k}, proportion = {k / n:.4f}") # calculate %

print(f"\nANSWER: {k / n:.1%} of Banking reviews about the app or website are negative "
      f"({k} of {n}).")

T1-E1  |  dataset: easy  |  answerable: yes
What proportion of Banking reviews that mention the app or website are negative?
----------------------------------------------------------------------------------------------------
sentiment
positive    237
neutral     110
negative     80

mentions = 427, negative = 80, proportion = 0.1874

ANSWER: 18.7% of Banking reviews about the app or website are negative (80 of 427).


In [8]:
80/427

0.1873536299765808

In [9]:
df = ask("T1-E2")

sub = df[(df.industry == "Fashion") & (df.child_aspect == "ease-of-use")]
n = len(sub)
k = sub.sentiment.eq("positive").sum()
print(sub.sentiment.value_counts().to_string())
print(f"\nmentions = {n}, positive = {k}, proportion = {k / n:.4f}")

print(f"\nANSWER: {k / n:.1%} of Fashion reviews about ease of use are positive "
      f"({k} of {n}).")

T1-E2  |  dataset: easy  |  answerable: yes
What proportion of Fashion reviews that mention ease of use are positive?
----------------------------------------------------------------------------------------------------
sentiment
positive    402
negative     60
neutral       1

mentions = 463, positive = 402, proportion = 0.8683

ANSWER: 86.8% of Fashion reviews about ease of use are positive (402 of 463).


In [10]:
402/(402+60+1)

0.8682505399568035

In [11]:
df = ask("T1-H1")

sub = df[(df.industry == "Fashion") & (df.child_aspect == "speed")]
n = len(sub)
k = sub.sentiment.eq("negative").sum()
print(sub.sentiment.value_counts().to_string())
print(f"\nmentions = {n}, negative = {k}, proportion = {k / n:.4f}")

print(f"\nANSWER: {k / n:.1%} of Fashion reviews about speed are negative "
      f"({k} of {n}).")

T1-H1  |  dataset: hard  |  answerable: yes
What proportion of Fashion reviews that mention speed are negative?
----------------------------------------------------------------------------------------------------
sentiment
positive    431
negative    117
neutral       1

mentions = 549, negative = 117, proportion = 0.2131

ANSWER: 21.3% of Fashion reviews about speed are negative (117 of 549).


In [12]:
117/(431+117+1)

0.21311475409836064

In [13]:
df = ask("T1-H2")

sub = df[(df.industry == "Price Comparison") & (df.child_aspect == "attitude-of-staff")]
n = len(sub)
k = sub.sentiment.eq("negative").sum()
print(sub.sentiment.value_counts().to_string())
print(f"\nmentions = {n}, negative = {k}, proportion = {k / n:.4f}")

print(f"\nANSWER: {k / n:.1%} of Price Comparison reviews about staff attitude are negative "
      f"({k} of {n}).")

T1-H2  |  dataset: hard  |  answerable: yes
What proportion of Price Comparison reviews that mention staff attitude are negative?
----------------------------------------------------------------------------------------------------
sentiment
positive    351
negative     43
neutral       2

mentions = 396, negative = 43, proportion = 0.1086

ANSWER: 10.9% of Price Comparison reviews about staff attitude are negative (43 of 396).


In [14]:
43/(351+43+2)

0.10858585858585859

In [15]:
# abstention: not enough reviews
df = ask("T1-H3")

sub = df[(df.industry == "Ride Hailing") & (df.child_aspect == "discounts-promotions")]
n = len(sub)
print(sub.sentiment.value_counts().to_string())
print(f"\nmentions = {n} (minimum required = {MIN_N})")

print(f"\nANSWER: No answer. Ride Hailing has only {n} reviews mentioning discounts and")
print(f"promotions (minimum {MIN_N}), so no proportion can be reported reliably.")

T1-H3  |  dataset: hard  |  answerable: no (abstain)
What proportion of Ride Hailing reviews that mention discounts and promotions are negative?
----------------------------------------------------------------------------------------------------
sentiment
negative    2
positive    1

mentions = 3 (minimum required = 30)

ANSWER: No answer. Ride Hailing has only 3 reviews mentioning discounts and
promotions (minimum 30), so no proportion can be reported reliably.


## T2 - Distribution

In [16]:
df = ask("T2-E1")

sub = df[df.industry == "Ride Hailing"] # filter
out = sub.sentiment.value_counts().to_frame("reviews")  # counts by sentiment
out["share"] = (out.reviews / len(sub)).round(4) # count %
print(out)
print(f"\ntotal Ride Hailing reviews = {len(sub)}")

# negative outweighs positive here, so the customer base reads as dissatisfied
print(f"\nANSWER: The customer base is mostly dissatisfied. Ride Hailing sentiment is "
      f"{out.share['positive']:.1%} positive, {out.share['negative']:.1%} negative and "
      f"{out.share['neutral']:.1%} neutral across {len(sub)} mentions.")

T2-E1  |  dataset: easy  |  answerable: yes
What is the sentiment breakdown across all Ride Hailing reviews?
----------------------------------------------------------------------------------------------------
           reviews   share
sentiment                 
negative      1115  0.7278
positive       338  0.2206
neutral         79  0.0516

total Ride Hailing reviews = 1532

ANSWER: The customer base is mostly dissatisfied. Ride Hailing sentiment is 22.1% positive, 72.8% negative and 5.2% neutral across 1532 mentions.


In [17]:
print(1115/(1115+338+79))
print(338/1532)
print(79/1532)
print(0.7278+0.2206+0.0516)

0.7278067885117493
0.2206266318537859
0.05156657963446475
1.0


In [18]:
df = ask("T2-E2")

sub = df[df.org == "Marbrook"]
out = sub.sentiment.value_counts().to_frame("reviews")
out["share"] = (out.reviews / len(sub)).round(4)
print(out)
print(f"\ntotal Marbrook reviews = {len(sub)}")

# positive outweighs negative here, so the customer base reads as satisfied
print(f"\nANSWER: The customer base is mostly satisfied. Marbrook sentiment is "
      f"{out.share['positive']:.1%} positive, {out.share['negative']:.1%} negative and "
      f"{out.share['neutral']:.1%} neutral across {len(sub)} mentions.")

T2-E2  |  dataset: easy  |  answerable: yes
What is the sentiment breakdown across all Marbrook reviews?
----------------------------------------------------------------------------------------------------
           reviews   share
sentiment                 
positive       685  0.6561
negative       342  0.3276
neutral         17  0.0163

total Marbrook reviews = 1044

ANSWER: The customer base is mostly satisfied. Marbrook sentiment is 65.6% positive, 32.8% negative and 1.6% neutral across 1044 mentions.


In [19]:
print(342/(685+342+17))
print(0.6561+0.3276+0.0163)

0.3275862068965517
1.0


In [20]:
df = ask("T2-H1")

sub = df[df.org == "Vanter Financial"]
out = sub.sentiment.value_counts().to_frame("reviews")
out["share"] = (out.reviews / len(sub)).round(4)
print(out)
print(f"\ntotal Vanter Financial reviews = {len(sub)}")

print(f"\nANSWER: The customer base is mostly satisfied. Vanter Financial sentiment is "
      f"{out.share['positive']:.1%} positive, {out.share['negative']:.1%} negative and "
      f"{out.share['neutral']:.1%} neutral across {len(sub)} mentions.")

T2-H1  |  dataset: hard  |  answerable: yes
What is the sentiment breakdown across all Vanter Financial reviews?
----------------------------------------------------------------------------------------------------
           reviews   share
sentiment                 
positive       253  0.6024
negative       119  0.2833
neutral         48  0.1143

total Vanter Financial reviews = 420

ANSWER: The customer base is mostly satisfied. Vanter Financial sentiment is 60.2% positive, 28.3% negative and 11.4% neutral across 420 mentions.


In [21]:
print(48/(253+119+48))
print(0.6024+0.2833+0.1143)

0.11428571428571428
1.0


In [22]:
df = ask("T2-H2")

sub = df[df.org == "PricePilot"]
out = sub.sentiment.value_counts().to_frame("reviews")
out["share"] = (out.reviews / len(sub)).round(4)
print(out)
print(f"\ntotal PricePilot reviews = {len(sub)}")

print(f"\nANSWER: The customer base is mostly satisfied. PricePilot sentiment is "
      f"{out.share['positive']:.1%} positive, {out.share['negative']:.1%} negative and "
      f"{out.share['neutral']:.1%} neutral across {len(sub)} mentions.")

T2-H2  |  dataset: hard  |  answerable: yes
What is the sentiment breakdown across all PricePilot reviews?
----------------------------------------------------------------------------------------------------
           reviews   share
sentiment                 
positive       677  0.8410
negative       125  0.1553
neutral          3  0.0037

total PricePilot reviews = 805

ANSWER: The customer base is mostly satisfied. PricePilot sentiment is 84.1% positive, 15.5% negative and 0.4% neutral across 805 mentions.


In [23]:
print(677/(677+125+3))
print(0.8410+0.1553+0.0037)

0.8409937888198757
1.0


In [24]:
df = ask("T2-H3")

sub = df[df.org == "Sable Row"]
out = sub.sentiment.value_counts().to_frame("reviews")
out["share"] = (out.reviews / len(sub)).round(4)
print(out)
print(f"\ntotal Sable Row reviews = {len(sub)}")

print(f"\nANSWER: The customer base is mostly satisfied. Sable Row sentiment is "
      f"{out.share['positive']:.1%} positive, {out.share['negative']:.1%} negative and "
      f"{out.share['neutral']:.1%} neutral across {len(sub)} mentions.")

T2-H3  |  dataset: hard  |  answerable: yes
What is the sentiment breakdown across all Sable Row reviews?
----------------------------------------------------------------------------------------------------
           reviews   share
sentiment                 
positive       643  0.6951
negative       279  0.3016
neutral          3  0.0032

total Sable Row reviews = 925

ANSWER: The customer base is mostly satisfied. Sable Row sentiment is 69.5% positive, 30.2% negative and 0.3% neutral across 925 mentions.


In [25]:
print(3/(643+279+3))
print(0.6951+0.3016+0.0032)

0.003243243243243243
0.9999


## T3 - Ranking by volume

In [26]:
df = ask("T3-E1")

neg = df[(df.industry == "Travel Booking") & (df.sentiment == "negative")] # filter
counts = neg.child_aspect.value_counts()  # sorted in descending order by default
print(counts.to_string())
print(f"\ntotal Travel Booking complaints = {len(neg)}")

print(f"\nANSWER: The top 3 topics with the most complaints in Travel Booking are: "
      f"the app or website ({counts['app-website']}), account access ({counts['account-access']}) "
      f"and email support ({counts['email']}).")

T3-E1  |  dataset: easy  |  answerable: yes
What are the top 3 topics with the most number of complaints in Travel Booking?
----------------------------------------------------------------------------------------------------
child_aspect
app-website              215
account-access           149
email                    138
phone                    116
general-satisfaction     103
attitude-of-staff         85
ease-of-use               82
discounts-promotions      74
competitor                54
reviews                   49
price-value-for-money     41
speed                     39

total Travel Booking complaints = 1145

ANSWER: The top 3 topics with the most complaints in Travel Booking are: the app or website (215), account access (149) and email support (138).


In [27]:
215+149+138+116+103+85+82+74+54+49+41+39

1145

In [28]:
df = ask("T3-E2")

neg = df[(df.industry == "Groceries") & (df.sentiment == "negative")]
counts = neg.child_aspect.value_counts()
print(counts.to_string())
print(f"\ntotal Groceries complaints = {len(neg)}")

print(f"\nANSWER: The top 3 topics with the most complaints in Groceries are: "
      f"the app or website ({counts['app-website']}), ease of use ({counts['ease-of-use']}) "
      f"and email support ({counts['email']}).")

T3-E2  |  dataset: easy  |  answerable: yes
What are the top 3 topics with the most number of complaints in Groceries?
----------------------------------------------------------------------------------------------------
child_aspect
app-website              383
ease-of-use              215
email                    100
reviews                   98
phone                     96
general-satisfaction      88
account-access            80
speed                     77
competitor                56
discounts-promotions      55
attitude-of-staff         29
price-value-for-money      9

total Groceries complaints = 1286

ANSWER: The top 3 topics with the most complaints in Groceries are: the app or website (383), ease of use (215) and email support (100).


In [29]:
df = ask("T3-H1")

neg = df[(df.industry == "Fashion") & (df.sentiment == "negative")]
counts = neg.child_aspect.value_counts()
print(counts.to_string())
print(f"\ntotal Fashion complaints = {len(neg)}")

print(f"\nANSWER: The top 3 topics with the most complaints in Fashion are: "
      f"the app or website ({counts['app-website']}), "
      f"general satisfaction ({counts['general-satisfaction']}) "
      f"and speed ({counts['speed']}).")

T3-H1  |  dataset: hard  |  answerable: yes
What are the top 3 topics with the most number of complaints in Fashion?
----------------------------------------------------------------------------------------------------
child_aspect
app-website              241
general-satisfaction     160
speed                    117
attitude-of-staff        116
phone                     70
price-value-for-money     49
ease-of-use               37
email                     30
discounts-promotions      26
account-access            20
reviews                   15
competitor                13

total Fashion complaints = 894

ANSWER: The top 3 topics with the most complaints in Fashion are: the app or website (241), general satisfaction (160) and speed (117).


In [30]:
df = ask("T3-H2")

neg = df[(df.industry == "Price Comparison") & (df.sentiment == "negative")]
counts = neg.child_aspect.value_counts()
print(counts.to_string())
print(f"\ntotal Price Comparison complaints = {len(neg)}")

print(f"\nANSWER: The top 3 topics with the most complaints in Price Comparison are: "
      f"the app or website ({counts['app-website']}), "
      f"price and value for money ({counts['price-value-for-money']}) "
      f"and general satisfaction ({counts['general-satisfaction']}).")

T3-H2  |  dataset: hard  |  answerable: yes
What are the top 3 topics with the most number of complaints in Price Comparison?
----------------------------------------------------------------------------------------------------
child_aspect
app-website              148
price-value-for-money     65
general-satisfaction      48
phone                     45
attitude-of-staff         43
discounts-promotions      33
ease-of-use               30
email                     27
account-access            26
competitor                24
speed                     13
reviews                    7

total Price Comparison complaints = 509

ANSWER: The top 3 topics with the most complaints in Price Comparison are: the app or website (148), price and value for money (65) and general satisfaction (48).


In [31]:
# abstention: not enough topics
df = ask("T3-H3")

sub = df[df.industry == "Consulting"]
print(sub.groupby("child_aspect").sentiment.value_counts().unstack(fill_value=0))
neg = sub[sub.sentiment == "negative"]
print(f"\nConsulting reviews = {len(sub)}, complaints = {len(neg)},"
      f" topics with any complaint = {neg.child_aspect.nunique()}")

print("\nANSWER: No answer. Consulting reviews in the dataset only cover one")
print("topic (account access), so a top 3 ranking of complaint topics does not exist.")

T3-H3  |  dataset: hard  |  answerable: no (abstain)
What are the top 3 topics with the most number of complaints in Consulting?
----------------------------------------------------------------------------------------------------
sentiment       negative  neutral  positive
child_aspect                               
account-access        52       61         4

Consulting reviews = 117, complaints = 52, topics with any complaint = 1

ANSWER: No answer. Consulting reviews in the dataset only cover one
topic (account access), so a top 3 ranking of complaint topics does not exist.


## T4 - Cross-segment comparison

In [32]:
df = ask("T4-E1")

sub = df[(df.child_aspect == "price-value-for-money")
         & (df.industry.isin(["Banking", "Ride Hailing"]))] # filter

# mentions / negatives / negative rate per industry
t = (sub.assign(is_neg=sub.sentiment.eq("negative"))
        .groupby("industry")
        .agg(mentions=("sentiment", "size"), negatives=("is_neg", "sum"))) # get negative and total counts per industry
t["neg_rate"] = (t.negatives / t.mentions).round(4) # calculate %
print(t)

print(f"\nANSWER: Ride Hailing has the higher negative rate ({t.neg_rate['Ride Hailing']:.1%}) "
      f"for price and value for money, compared to {t.neg_rate['Banking']:.1%} for Banking.")

T4-E1  |  dataset: easy  |  answerable: yes
Is negative sentiment about price and value for money higher in Banking or Ride Hailing?
----------------------------------------------------------------------------------------------------


              mentions  negatives  neg_rate
industry                                   
Banking            160         32    0.2000
Ride Hailing       120         91    0.7583

ANSWER: Ride Hailing has the higher negative rate (75.8%) for price and value for money, compared to 20.0% for Banking.


In [33]:
print(32/160)
print(91/120)

0.2
0.7583333333333333


In [34]:
df = ask("T4-E2")

sub = df[(df.child_aspect == "app-website")
         & (df.industry.isin(["Fashion", "Travel Booking"]))]

t = (sub.assign(is_neg=sub.sentiment.eq("negative"))
        .groupby("industry")
        .agg(mentions=("sentiment", "size"), negatives=("is_neg", "sum")))
t["neg_rate"] = (t.negatives / t.mentions).round(4)
print(t)

print(f"\nANSWER: Travel Booking has the higher negative rate ({t.neg_rate['Travel Booking']:.1%}) "
      f"for the app or website, compared to {t.neg_rate['Fashion']:.1%} for Fashion.")

T4-E2  |  dataset: easy  |  answerable: yes
Is negative sentiment about the app or website higher in Fashion or Travel Booking?
----------------------------------------------------------------------------------------------------
                mentions  negatives  neg_rate
industry                                     
Fashion              788        220    0.2792
Travel Booking       488        215    0.4406

ANSWER: Travel Booking has the higher negative rate (44.1%) for the app or website, compared to 27.9% for Fashion.


In [35]:
215/488

0.4405737704918033

In [36]:
df = ask("T4-H1")

sub = df[(df.child_aspect == "ease-of-use")
         & (df.industry.isin(["Groceries", "Trading"]))]

t = (sub.assign(is_neg=sub.sentiment.eq("negative"))
        .groupby("industry")
        .agg(mentions=("sentiment", "size"), negatives=("is_neg", "sum")))
t["neg_rate"] = (t.negatives / t.mentions).round(4)
print(t)

print(f"\nANSWER: Groceries has the higher negative rate ({t.neg_rate['Groceries']:.1%}) "
      f"for ease of use, compared to {t.neg_rate['Trading']:.1%} for Trading.")

T4-H1  |  dataset: hard  |  answerable: yes
Is negative sentiment about ease of use higher in Groceries or Trading?
----------------------------------------------------------------------------------------------------


           mentions  negatives  neg_rate
industry                                
Groceries       523        222    0.4245
Trading         267         29    0.1086

ANSWER: Groceries has the higher negative rate (42.4%) for ease of use, compared to 10.9% for Trading.


In [37]:
222/523

0.42447418738049714

In [38]:
# abstention: 1 industry has no reviews
df = ask("T4-H2")

sub = df[(df.child_aspect == "ease-of-use")
         & (df.industry.isin(["Banking", "Consulting"]))]

t = (sub.assign(is_neg=sub.sentiment.eq("negative"))
        .groupby("industry")
        .agg(mentions=("sentiment", "size"), negatives=("is_neg", "sum")))
t["neg_rate"] = (t.negatives / t.mentions).round(4)
print(t.reindex(["Banking", "Consulting"]))
print(f"\nBanking mentions = {(sub.industry == 'Banking').sum()},"
      f" Consulting mentions = {(sub.industry == 'Consulting').sum()}")

print("\nANSWER: No answer. Consulting has no reviews mentioning ease of use in the")
print("dataset, so the two industries cannot be compared on this topic.")

T4-H2  |  dataset: hard  |  answerable: no (abstain)
Is negative sentiment about ease of use higher in Banking or Consulting?
----------------------------------------------------------------------------------------------------
            mentions  negatives  neg_rate
industry                                 
Banking        383.0       96.0    0.2507
Consulting       NaN        NaN       NaN

Banking mentions = 383, Consulting mentions = 0

ANSWER: No answer. Consulting has no reviews mentioning ease of use in the
dataset, so the two industries cannot be compared on this topic.


In [39]:
# abstention: both industries have no reviews
df = ask("T4-H3")

sub = df[(df.child_aspect == "price-value-for-money")
         & (df.industry.isin(["Consulting", "Streaming"]))]

t = (sub.assign(is_neg=sub.sentiment.eq("negative"))
        .groupby("industry")
        .agg(mentions=("sentiment", "size"), negatives=("is_neg", "sum")))
t["neg_rate"] = (t.negatives / t.mentions).round(4)
print(t.reindex(["Consulting", "Streaming"]))
print(f"\nConsulting mentions = {(sub.industry == 'Consulting').sum()},"
      f" Streaming mentions = {(sub.industry == 'Streaming').sum()}")

print("\nANSWER: No answer. Neither Consulting nor Streaming has any reviews mentioning price and")
print("value for money in the dataset, so the two industries cannot be compared on this topic.")

T4-H3  |  dataset: hard  |  answerable: no (abstain)
Is negative sentiment about price and value for money higher in Consulting or Streaming?
----------------------------------------------------------------------------------------------------
            mentions  negatives  neg_rate
industry                                 
Consulting       NaN        NaN       NaN
Streaming        NaN        NaN       NaN

Consulting mentions = 0, Streaming mentions = 0

ANSWER: No answer. Neither Consulting nor Streaming has any reviews mentioning price and
value for money in the dataset, so the two industries cannot be compared on this topic.


## T5 - Statistical testing

In [40]:
df = ask("T5-E1")

sub = df[(df.child_aspect == "attitude-of-staff")
         & (df.org.isin(["CompareHive", "Tallywise"]))]  # filter

t = (sub.assign(is_neg=sub.sentiment.eq("negative"))
        .groupby("org")
        .agg(mentions=("sentiment", "size"), negatives=("is_neg", "sum")))  # get total and negative counts per org
t["neg_rate"] = (t.negatives / t.mentions).round(4)  # calculate negative rate
print(t.reindex(["CompareHive", "Tallywise"]))

# check first whether the total mentions for each org is at least 30

# two-sided two-proportion z-test with pooled variance
k1, n1 = t.negatives["CompareHive"], t.mentions["CompareHive"]
k2, n2 = t.negatives["Tallywise"], t.mentions["Tallywise"]
p_pool = (k1 + k2) / (n1 + n2)
z = (k1 / n1 - k2 / n2) / np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
p = 2 * stats.norm.sf(abs(z))
print(f"\nz = {z:.3f}, p = {p:.4f}")

print(f"\nANSWER: No. The difference in negative rates is not statistically "
      f"significant (p = {p:.3f}).")

T5-E1  |  dataset: easy  |  answerable: yes
Do CompareHive and Tallywise have significantly different negative rates for staff attitude?
----------------------------------------------------------------------------------------------------
             mentions  negatives  neg_rate
org                                       
CompareHive       103         11    0.1068
Tallywise         118         18    0.1525

z = -1.005, p = 0.3150

ANSWER: No. The difference in negative rates is not statistically significant (p = 0.315).


In [41]:
df = ask("T5-E2")

sub = df[(df.child_aspect == "general-satisfaction")
         & (df.org.isin(["Trippa", "Roamly"]))]

t = (sub.assign(is_neg=sub.sentiment.eq("negative"))
        .groupby("org")
        .agg(mentions=("sentiment", "size"), negatives=("is_neg", "sum")))
t["neg_rate"] = (t.negatives / t.mentions).round(4)
print(t.reindex(["Trippa", "Roamly"]))

k1, n1 = t.negatives["Trippa"], t.mentions["Trippa"]
k2, n2 = t.negatives["Roamly"], t.mentions["Roamly"]
p_pool = (k1 + k2) / (n1 + n2)
z = (k1 / n1 - k2 / n2) / np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
p = 2 * stats.norm.sf(abs(z))
print(f"\nz = {z:.3f}, p = {p:.4f}")

# p < 0.05 and Trippa is the worse of the two
print(f"\nANSWER: Yes, Trippa performs worse. Trippa's negative rate for general satisfaction is "
      f"{k1 / n1:.1%} (n={n1}) against {k2 / n2:.1%} (n={n2}) for Roamly. The difference is "
      f"statistically significant (p < 0.001).")

T5-E2  |  dataset: easy  |  answerable: yes
Do Trippa and Roamly have significantly different negative rates for general satisfaction?
----------------------------------------------------------------------------------------------------
        mentions  negatives  neg_rate
org                                  
Trippa       123         54    0.4390
Roamly        95         21    0.2211

z = 3.359, p = 0.0008

ANSWER: Yes, Trippa performs worse. Trippa's negative rate for general satisfaction is 43.9% (n=123) against 22.1% (n=95) for Roamly. The difference is statistically significant (p < 0.001).


In [42]:
df = ask("T5-H1")

sub = df[(df.child_aspect == "email")
         & (df.org.isin(["Swiftly Rides", "Kerbside"]))]

t = (sub.assign(is_neg=sub.sentiment.eq("negative"))
        .groupby("org")
        .agg(mentions=("sentiment", "size"), negatives=("is_neg", "sum")))
t["neg_rate"] = (t.negatives / t.mentions).round(4)
print(t.reindex(["Swiftly Rides", "Kerbside"]))

k1, n1 = t.negatives["Swiftly Rides"], t.mentions["Swiftly Rides"]
k2, n2 = t.negatives["Kerbside"], t.mentions["Kerbside"]
p_pool = (k1 + k2) / (n1 + n2)
z = (k1 / n1 - k2 / n2) / np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
p = 2 * stats.norm.sf(abs(z))
print(f"\nz = {z:.3f}, p = {p:.4f}")

print(f"\nANSWER: No. The difference in negative rates is not statistically "
      f"significant (p = {p:.3f}).")

T5-H1  |  dataset: hard  |  answerable: yes
Do Swiftly Rides and Kerbside have significantly different negative rates for email support?
----------------------------------------------------------------------------------------------------
               mentions  negatives  neg_rate
org                                         
Swiftly Rides        40         28     0.700
Kerbside             40         31     0.775

z = -0.762, p = 0.4459

ANSWER: No. The difference in negative rates is not statistically significant (p = 0.446).


In [43]:
# abstention: insufficient data
df = ask("T5-H2")

sub = df[(df.child_aspect == "price-value-for-money")
         & (df.org.isin(["Halden Savings", "Kestrel Bank"]))]

t = (sub.assign(is_neg=sub.sentiment.eq("negative"))
        .groupby("org")
        .agg(mentions=("sentiment", "size"), negatives=("is_neg", "sum")))
t["neg_rate"] = (t.negatives / t.mentions).round(4)
print(t.reindex(["Halden Savings", "Kestrel Bank"]))
print(f"\nminimum required per organisation = {MIN_N}")

print(f"\nANSWER: No answer. Both organisations have less than 30 reviews for price "
      "and value")
print("for money, so a two-proportion test on these counts would not be reliable.")

T5-H2  |  dataset: hard  |  answerable: no (abstain)
Do Halden Savings and Kestrel Bank have significantly different negative rates for price and value for money?
----------------------------------------------------------------------------------------------------
                mentions  negatives  neg_rate
org                                          
Halden Savings        29         16    0.5517
Kestrel Bank          20          6    0.3000

minimum required per organisation = 30

ANSWER: No answer. Both organisations have less than 30 reviews for price and value
for money, so a two-proportion test on these counts would not be reliable.


In [44]:
# abstention: insufficient data
df = ask("T5-H3")

sub = df[(df.child_aspect == "speed")
         & (df.org.isin(["Northpeak Trading", "Quantly"]))]

t = (sub.assign(is_neg=sub.sentiment.eq("negative"))
        .groupby("org")
        .agg(mentions=("sentiment", "size"), negatives=("is_neg", "sum")))
t["neg_rate"] = (t.negatives / t.mentions).round(4)
print(t.reindex(["Northpeak Trading", "Quantly"]))
print(f"\nminimum required per organisation = {MIN_N}")

print(f"\nANSWER: No answer. Both organisations have less than 30 reviews for price "
      "and value")
print("for money, so a two-proportion test on these counts would not be reliable.")

T5-H3  |  dataset: hard  |  answerable: no (abstain)
Do Northpeak Trading and Quantly have significantly different negative rates for speed?
----------------------------------------------------------------------------------------------------
                   mentions  negatives  neg_rate
org                                             
Northpeak Trading        20         12      0.60
Quantly                  25         13      0.52

minimum required per organisation = 30

ANSWER: No answer. Both organisations have less than 30 reviews for price and value
for money, so a two-proportion test on these counts would not be reliable.


## T6 - Organisation vs industry comparison

In [45]:
df = ask("T6-E1")

sub = df[(df.industry == "Banking") & (df.child_aspect == "app-website")] # filter
industry_rate = sub.sentiment.eq("negative").mean() # get industry average
print(f"Banking industry average negative rate on the app or website = "
      f"{industry_rate:.1%} (n = {len(sub)})\n")

# get negative rate per org
t = (sub.assign(is_neg=sub.sentiment.eq("negative"))
        .groupby("org")
        .agg(mentions=("sentiment", "size"), negatives=("is_neg", "sum")))

# drop orgs with less than 30 reviews
kept = t[t.mentions >= MIN_N] 
kept["neg_rate"] = (kept.negatives / kept.mentions).round(4) # calculate negative rate
kept["vs_industry_pp"] = ((kept.neg_rate - industry_rate) * 100).round(2) # calculate difference with industry rate
kept = kept.sort_values("neg_rate", ascending=False)

print(f"\nafter dropping organisations with < {MIN_N} mentions:"); print(kept)

print(f"\nANSWER: Negative rates for the app or website vary from {kept.neg_rate.min():.1%} to "
      f"{kept.neg_rate.max():.1%} across the {len(kept)} organisations with at least {MIN_N} "
      f"mentions. The industry average is {industry_rate:.1%}. "
      f"Halden Savings ({kept.neg_rate['Halden Savings']:.1%}, n={kept.mentions['Halden Savings']}) "
      f"and Northeast Bank ({kept.neg_rate['Northeast Bank']:.1%}, n={kept.mentions['Northeast Bank']}) "
      f"are above average, whereas "
      f"Vanter Financial ({kept.neg_rate['Vanter Financial']:.1%}, n={kept.mentions['Vanter Financial']}) "
      f"and Kestrel Bank ({kept.neg_rate['Kestrel Bank']:.1%}, n={kept.mentions['Kestrel Bank']}) "
      f"are below.")

T6-E1  |  dataset: easy  |  answerable: yes
How do individual organisations in Banking compare to the industry average on the app or website? Give me each organisation's negative rate and ignore any organisations with less than 30 mentions about the app or website.
----------------------------------------------------------------------------------------------------
Banking industry average negative rate on the app or website = 18.7% (n = 427)




after dropping organisations with < 30 mentions:
                  mentions  negatives  neg_rate  vs_industry_pp
org                                                            
Halden Savings          62         24    0.3871           19.97
Northeast Bank         140         28    0.2000            1.26
Vanter Financial        86         11    0.1279           -5.95
Kestrel Bank           139         17    0.1223           -6.51

ANSWER: Negative rates for the app or website vary from 12.2% to 38.7% across the 4 organisations with at least 30 mentions. The industry average is 18.7%. Halden Savings (38.7%, n=62) and Northeast Bank (20.0%, n=140) are above average, whereas Vanter Financial (12.8%, n=86) and Kestrel Bank (12.2%, n=139) are below.


In [46]:
df = ask("T6-E2")

sub = df[(df.industry == "Travel Booking") & (df.child_aspect == "general-satisfaction")]
industry_rate = sub.sentiment.eq("negative").mean()

print(f"Travel Booking industry average negative rate on general satisfaction = "
      f"{industry_rate:.1%} (n = {len(sub)})")

t = (sub.assign(is_neg=sub.sentiment.eq("negative"))
        .groupby("org")
        .agg(mentions=("sentiment", "size"), negatives=("is_neg", "sum")))

kept = t[t.mentions >= MIN_N] 
kept["neg_rate"] = (kept.negatives / kept.mentions).round(4) 
kept["vs_industry_pp"] = ((kept.neg_rate - industry_rate) * 100).round(2) 
kept = kept.sort_values("neg_rate", ascending=False)

print(f"\nafter dropping organisations with < {MIN_N} mentions:"); print(kept)

print(f"\nANSWER: Negative rates for general satisfaction vary from {kept.neg_rate.min():.1%} to "
      f"{kept.neg_rate.max():.1%} across the {len(kept)} organisations with at least {MIN_N} "
      f"mentions. The industry average is {industry_rate:.1%}. "
      f"Trippa ({kept.neg_rate['Trippa']:.1%}, n={kept.mentions['Trippa']}) is above average, "
      f"whereas Roamly ({kept.neg_rate['Roamly']:.1%}, n={kept.mentions['Roamly']}), "
      f"Journeo ({kept.neg_rate['Journeo']:.1%}, n={kept.mentions['Journeo']}) "
      f"and Wayfare ({kept.neg_rate['Wayfare']:.1%}, n={kept.mentions['Wayfare']}) are below.")

T6-E2  |  dataset: easy  |  answerable: yes
How do individual organisations in Travel Booking compare to the industry average on general satisfaction? Give me each organisation's negative rate and ignore any organisations with less than 30 mentions about general satisfaction.
----------------------------------------------------------------------------------------------------
Travel Booking industry average negative rate on general satisfaction = 25.5% (n = 404)

after dropping organisations with < 30 mentions:
         mentions  negatives  neg_rate  vs_industry_pp
org                                                   
Trippa        123         54    0.4390           18.40
Roamly         95         21    0.2211           -3.39
Journeo       107         21    0.1963           -5.87
Wayfare        79          7    0.0886          -16.64

ANSWER: Negative rates for general satisfaction vary from 8.9% to 43.9% across the 4 organisations with at least 30 mentions. The industry average is 25.

In [47]:
df = ask("T6-H1")

sub = df[(df.industry == "Fashion") & (df.child_aspect == "app-website")]
industry_rate = sub.sentiment.eq("negative").mean()

print(f"Fashion industry average negative rate on the app or website = "
      f"{industry_rate:.1%} (n = {len(sub)})")

t = (sub.assign(is_neg=sub.sentiment.eq("negative"))
        .groupby("org")
        .agg(mentions=("sentiment", "size"), negatives=("is_neg", "sum")))

kept = t[t.mentions >= MIN_N] 
kept["neg_rate"] = (kept.negatives / kept.mentions).round(4) 
kept["vs_industry_pp"] = ((kept.neg_rate - industry_rate) * 100).round(2) 
kept = kept.sort_values("neg_rate", ascending=False)

print(f"\nafter dropping organisations with < {MIN_N} mentions:"); print(kept)

print(f"\nANSWER: Negative rates for the app or website vary from {kept.neg_rate.min():.1%} to "
      f"{kept.neg_rate.max():.1%} across the {len(kept)} organisations with at least {MIN_N} "
      f"mentions. The industry average is {industry_rate:.1%}. "
      f"Marbrook ({kept.neg_rate['Marbrook']:.1%}, n={kept.mentions['Marbrook']}) "
      f"and Northerly ({kept.neg_rate['Northerly']:.1%}, n={kept.mentions['Northerly']}) "
      f"are above average, whereas "
      f"Vella & Co ({kept.neg_rate['Vella & Co']:.1%}, n={kept.mentions['Vella & Co']}) "
      f"and Sable Row ({kept.neg_rate['Sable Row']:.1%}, n={kept.mentions['Sable Row']}) "
      f"are below.")

T6-H1  |  dataset: hard  |  answerable: yes
How do individual organisations in Fashion compare to the industry average on the app or website? Give me each organisation's negative rate and ignore any organisations with less than 30 mentions about the app or website.
----------------------------------------------------------------------------------------------------
Fashion industry average negative rate on the app or website = 27.5% (n = 877)

after dropping organisations with < 30 mentions:
            mentions  negatives  neg_rate  vs_industry_pp
org                                                      
Marbrook         276        103    0.3732            9.84
Northerly        192         63    0.3281            5.33
Vella & Co       156         29    0.1859           -8.89
Sable Row        253         46    0.1818           -9.30

ANSWER: Negative rates for the app or website vary from 18.2% to 37.3% across the 4 organisations with at least 30 mentions. The industry average is 27.5%.

In [48]:
# abstention: the industry has no reviews on this topic
df = ask("T6-H2")

sub = df[(df.industry == "Consulting") & (df.child_aspect == "app-website")]
print(f"Consulting reviews mentioning the app or website = {len(sub)}")
print("Consulting topics present in the dataset:")
print(df[df.industry == "Consulting"].child_aspect.value_counts().to_string())

print("\nANSWER: No answer. No Consulting review mentions the app or website in the dataset,")
print("so no comparison can be made.")

T6-H2  |  dataset: hard  |  answerable: no (abstain)
How do individual organisations in Consulting compare to the industry average on the app or website? Give me each organisation's negative rate and ignore any organisations with less than 30 mentions about the app or website.
----------------------------------------------------------------------------------------------------
Consulting reviews mentioning the app or website = 0
Consulting topics present in the dataset:
child_aspect
account-access    117

ANSWER: No answer. No Consulting review mentions the app or website in the dataset,
so no comparison can be made.


In [49]:
# abstention: the industry has no reviews on this topic
df = ask("T6-H3")

sub = df[(df.industry == "Streaming") & (df.child_aspect == "general-satisfaction")]
print(f"Streaming reviews mentioning general satisfaction = {len(sub)}")
print("Streaming topics present in the dataset:")
print(df[df.industry == "Streaming"].child_aspect.value_counts().to_string())

print("\nANSWER: No answer. No Streaming review describes general satisfaction in the dataset,")
print("so no comparison can be made.")

T6-H3  |  dataset: hard  |  answerable: no (abstain)
How do individual organisations in Streaming compare to the industry average on general satisfaction? Give me each organisation's negative rate and ignore any organisations with less than 30 mentions about general satisfaction.
----------------------------------------------------------------------------------------------------
Streaming reviews mentioning general satisfaction = 0
Streaming topics present in the dataset:
child_aspect
account-access          89
app-website             46
discounts-promotions    11

ANSWER: No answer. No Streaming review describes general satisfaction in the dataset,
so no comparison can be made.


## T7 - Driver identification by severity

In [50]:
df = ask("T7-E1")

sub = df[df.org == "Northeast Bank"] # filter
overall_rate = sub.sentiment.eq("negative").mean()

t = (sub.assign(is_neg=sub.sentiment.eq("negative"))
        .groupby("child_aspect")
        .agg(mentions=("sentiment", "size"), negatives=("is_neg", "sum")))
t["neg_rate"] = (t.negatives / t.mentions).round(4)
t = t.sort_values("neg_rate", ascending=False)

# drop topics with less than 30 reviews
kept = t[t.mentions >= MIN_N]
print(f"\nranked, topics with >= {MIN_N} mentions:"); print(kept)
print(f"\nNortheast Bank overall negative rate = {overall_rate:.1%} ({len(sub)} reviews)")

# the worst topic sits 49.6 pp above the overall rate, so "well above"
print(f"\nANSWER: Email support is Northeast Bank's biggest driver of dissatisfaction, with an "
      f"{kept.neg_rate['email']:.1%} negative rate across {kept.mentions['email']} mentions. "
      f"This is well above its overall negative rate of {overall_rate:.1%}.")

T7-E1  |  dataset: easy  |  answerable: yes
What is the biggest driver of dissatisfaction at Northeast Bank? Rank topics by negative rate and ignore any topics with less than 30 mentions.
----------------------------------------------------------------------------------------------------

ranked, topics with >= 30 mentions:
                       mentions  negatives  neg_rate
child_aspect                                        
email                        40         35    0.8750
discounts-promotions         40         33    0.8250
attitude-of-staff            40         31    0.7750
account-access               40         25    0.6250
reviews                      40         25    0.6250
phone                        40         21    0.5250
ease-of-use                  57         20    0.3509
competitor                   40         10    0.2500
app-website                 140         28    0.2000
speed                        40          6    0.1500
general-satisfaction         85       

In [51]:
df = ask("T7-E2")

sub = df[df.org == "Wayfare"]
overall_rate = sub.sentiment.eq("negative").mean()

t = (sub.assign(is_neg=sub.sentiment.eq("negative"))
        .groupby("child_aspect")
        .agg(mentions=("sentiment", "size"), negatives=("is_neg", "sum")))
t["neg_rate"] = (t.negatives / t.mentions).round(4)
t = t.sort_values("neg_rate", ascending=False)

kept = t[t.mentions >= MIN_N]
print(f"\nranked, topics with >= {MIN_N} mentions:"); print(kept)
print(f"\nWayfare overall negative rate = {overall_rate:.1%} ({len(sub)} reviews)")

print(f"\nANSWER: Account access is Wayfare's biggest driver of dissatisfaction, with a "
      f"{kept.neg_rate['account-access']:.1%} negative rate across "
      f"{kept.mentions['account-access']} mentions. "
      f"This is well above its overall negative rate of {overall_rate:.1%}.")

T7-E2  |  dataset: easy  |  answerable: yes
What is the biggest driver of dissatisfaction at Wayfare? Rank topics by negative rate and ignore any topics with less than 30 mentions.
----------------------------------------------------------------------------------------------------



ranked, topics with >= 30 mentions:
                       mentions  negatives  neg_rate
child_aspect                                        
account-access               40         37    0.9250
email                        40         34    0.8500
phone                        40         27    0.6750
attitude-of-staff            40         24    0.6000
discounts-promotions         40         22    0.5500
app-website                 159         82    0.5157
reviews                      40         15    0.3750
ease-of-use                  46         16    0.3478
speed                        40          8    0.2000
competitor                   40          7    0.1750
price-value-for-money        40          6    0.1500
general-satisfaction         79          7    0.0886

Wayfare overall negative rate = 44.3% (644 reviews)

ANSWER: Account access is Wayfare's biggest driver of dissatisfaction, with a 92.5% negative rate across 40 mentions. This is well above its overall negative rate of 4

In [52]:
df = ask("T7-H1")

sub = df[df.org == "Sable Row"]
overall_rate = sub.sentiment.eq("negative").mean()

t = (sub.assign(is_neg=sub.sentiment.eq("negative"))
        .groupby("child_aspect")
        .agg(mentions=("sentiment", "size"), negatives=("is_neg", "sum")))
t["neg_rate"] = (t.negatives / t.mentions).round(4)
t = t.sort_values("neg_rate", ascending=False)

kept = t[t.mentions >= MIN_N]
print(f"\nranked, topics with >= {MIN_N} mentions:"); print(kept)
print(f"\nSable Row overall negative rate = {overall_rate:.1%} ({len(sub)} reviews)")

print(f"\nANSWER: Staff attitude is Sable Row's biggest driver of dissatisfaction, with a "
      f"{kept.neg_rate['attitude-of-staff']:.1%} negative rate across "
      f"{kept.mentions['attitude-of-staff']} mentions. "
      f"This is well above its overall negative rate of {overall_rate:.1%}.")

T7-H1  |  dataset: hard  |  answerable: yes
What is the biggest driver of dissatisfaction at Sable Row? Rank topics by negative rate and ignore any topics with less than 30 mentions.
----------------------------------------------------------------------------------------------------



ranked, topics with >= 30 mentions:
                       mentions  negatives  neg_rate
child_aspect                                        
attitude-of-staff            56         44    0.7857
general-satisfaction        354        109    0.3079
speed                       100         30    0.3000
ease-of-use                  74         21    0.2838
price-value-for-money        40          8    0.2000
app-website                 253         46    0.1818

Sable Row overall negative rate = 30.2% (925 reviews)

ANSWER: Staff attitude is Sable Row's biggest driver of dissatisfaction, with a 78.6% negative rate across 56 mentions. This is well above its overall negative rate of 30.2%.


In [53]:
df = ask("T7-H2")

sub = df[df.org == "Investa"]
overall_rate = sub.sentiment.eq("negative").mean()

t = (sub.assign(is_neg=sub.sentiment.eq("negative"))
        .groupby("child_aspect")
        .agg(mentions=("sentiment", "size"), negatives=("is_neg", "sum")))
t["neg_rate"] = (t.negatives / t.mentions).round(4)
t = t.sort_values("neg_rate", ascending=False)

kept = t[t.mentions >= MIN_N]
print(f"\nranked, topics with >= {MIN_N} mentions:"); print(kept)
print(f"\nInvesta overall negative rate = {overall_rate:.1%} ({len(sub)} reviews)")

print(f"\nANSWER: Phone support is Investa's biggest driver of dissatisfaction, with a "
      f"{kept.neg_rate['phone']:.1%} negative rate across {kept.mentions['phone']} mentions. "
      f"This is well above its overall negative rate of {overall_rate:.1%}.")

T7-H2  |  dataset: hard  |  answerable: yes
What is the biggest driver of dissatisfaction at Investa? Rank topics by negative rate and ignore any topics with less than 30 mentions.
----------------------------------------------------------------------------------------------------

ranked, topics with >= 30 mentions:
                       mentions  negatives  neg_rate
child_aspect                                        
phone                        40         21    0.5250
price-value-for-money        34         14    0.4118
attitude-of-staff            54         17    0.3148
general-satisfaction         93         20    0.2151
app-website                 136         21    0.1544
ease-of-use                 155         18    0.1161

Investa overall negative rate = 25.6% (566 reviews)



ANSWER: Phone support is Investa's biggest driver of dissatisfaction, with a 52.5% negative rate across 40 mentions. This is well above its overall negative rate of 25.6%.


In [54]:
# abstention: insufficient data
df = ask("T7-H3")

sub = df[df.org == "Pinecast"]
t = (sub.assign(is_neg=sub.sentiment.eq("negative"))
        .groupby("child_aspect")
        .agg(mentions=("sentiment", "size"), negatives=("is_neg", "sum")))
t["neg_rate"] = (t.negatives / t.mentions).round(4)
t = t.sort_values("neg_rate", ascending=False)

print("all topics:"); print(t)
print(f"\nPinecast reviews = {len(sub)}, largest topic = {t.mentions.max()} mentions, "
      f"floor = {MIN_N}")

print(f"\nANSWER: No answer. All topics have too few reviews to claim that any of them drives dissatisfaction.")

T7-H3  |  dataset: hard  |  answerable: no (abstain)
What is the biggest driver of dissatisfaction at Pinecast? Rank topics by negative rate and ignore any topics with less than 30 mentions.
----------------------------------------------------------------------------------------------------


all topics:
                      mentions  negatives  neg_rate
child_aspect                                       
account-access               9          4    0.4444
discounts-promotions         4          1    0.2500
app-website                 15          3    0.2000

Pinecast reviews = 28, largest topic = 15 mentions, floor = 30

ANSWER: No answer. All topics have too few reviews to claim that any of them drives dissatisfaction.


## T8 - Prioritisation by a named rule

In [140]:
df = ask("T8-E1")

# 1. filter to the organisation and count by topic, with the negative rate per topic
d = df[df.org == "Wayfare"]
topics = (d.groupby("child_aspect")
            .agg(mentions=("sentiment", "size"),
                 negatives=("sentiment", lambda s: (s == "negative").sum()))
            .assign(neg_rate=lambda x: x.negatives / x.mentions)
            .sort_values("neg_rate", ascending=False))

# 2. drop the topics below the mention floor, and the non-actionable ones because the
#    answer is prescriptive
t = topics.query("mentions >= @MIN_N and child_aspect not in @NON_ACTIONABLE").copy()

# 3. Priority = 0.5 * minmax(negative rate) + 0.5 * minmax(volume), as the question defines
#    it: min-max puts the two figures on a common 0-1 scale so they are comparable
mm = lambda s: (s - s.min()) / (s.max() - s.min())
t["rate_scaled"] = mm(t.neg_rate)
t["vol_scaled"] = mm(t.mentions)
t["priority"] = 0.5 * t.rate_scaled + 0.5 * t.vol_scaled

# 4. rank by priority and take the top 2
t = t.sort_values("priority", ascending=False)
top2 = t.head(2)
print(t[["mentions", "neg_rate", "priority"]].round(3))

print(f"\nANSWER: Wayfare should address the app or website first "
      f"({top2.neg_rate['app-website']:.1%} negative rate across "
      f"{top2.mentions['app-website']} mentions), followed by account access "
      f"({top2.neg_rate['account-access']:.1%} across {top2.mentions['account-access']}).")

T8-E1  |  dataset: easy  |  answerable: yes
Which 2 issues should Wayfare address first? Consider both negative rate and volume equally, and ignore any topics with fewer than 30 mentions.
----------------------------------------------------------------------------------------------------
                       mentions  neg_rate  priority
child_aspect                                       
app-website                 159     0.516     0.736
account-access               40     0.925     0.500
email                        40     0.850     0.452
phone                        40     0.675     0.339
attitude-of-staff            40     0.600     0.290
discounts-promotions         40     0.550     0.258
ease-of-use                  46     0.348     0.153
speed                        40     0.200     0.032
price-value-for-money        40     0.150     0.000

ANSWER: Wayfare should address the app or website first (51.6% negative rate across 159 mentions), followed by account access (92.5% acros

In [141]:
df = ask("T8-E2")

d = df[df.org == "Larkmead Market"]
topics = (d.groupby("child_aspect")
            .agg(mentions=("sentiment", "size"),
                 negatives=("sentiment", lambda s: (s == "negative").sum()))
            .assign(neg_rate=lambda x: x.negatives / x.mentions)
            .sort_values("neg_rate", ascending=False))

t = topics.query("mentions >= @MIN_N and child_aspect not in @NON_ACTIONABLE").copy()

mm = lambda s: (s - s.min()) / (s.max() - s.min())
t["rate_scaled"] = mm(t.neg_rate)
t["vol_scaled"] = mm(t.mentions)
t["priority"] = 0.5 * t.rate_scaled + 0.5 * t.vol_scaled

t = t.sort_values("priority", ascending=False)
top2 = t.head(2)
print(t[["mentions", "neg_rate", "priority"]].round(3))

print(f"\nANSWER: Larkmead Market should address the app or website first "
      f"({top2.neg_rate['app-website']:.1%} negative rate across "
      f"{top2.mentions['app-website']} mentions), followed by ease of use "
      f"({top2.neg_rate['ease-of-use']:.1%} across {top2.mentions['ease-of-use']}).")

T8-E2  |  dataset: easy  |  answerable: yes
Which 2 issues should Larkmead Market address first? Consider both negative rate and volume equally, and ignore any topics with fewer than 30 mentions.
----------------------------------------------------------------------------------------------------
                       mentions  neg_rate  priority
child_aspect                                       
app-website                 259     0.479     0.737
ease-of-use                 175     0.617     0.631
phone                        40     0.900     0.500
email                        40     0.825     0.453
account-access               40     0.725     0.391
speed                        40     0.625     0.328
discounts-promotions         40     0.350     0.156
attitude-of-staff            40     0.225     0.078
price-value-for-money        40     0.100     0.000

ANSWER: Larkmead Market should address the app or website first (47.9% negative rate across 259 mentions), followed by ease of use

In [142]:
df = ask("T8-H1")

d = df[df.org == "Northerly"]
topics = (d.groupby("child_aspect")
            .agg(mentions=("sentiment", "size"),
                 negatives=("sentiment", lambda s: (s == "negative").sum()))
            .assign(neg_rate=lambda x: x.negatives / x.mentions)
            .sort_values("neg_rate", ascending=False))

t = topics.query("mentions >= @MIN_N and child_aspect not in @NON_ACTIONABLE").copy()

mm = lambda s: (s - s.min()) / (s.max() - s.min())
t["rate_scaled"] = mm(t.neg_rate)
t["vol_scaled"] = mm(t.mentions)
t["priority"] = 0.5 * t.rate_scaled + 0.5 * t.vol_scaled

t = t.sort_values("priority", ascending=False)
top2 = t.head(2)
print(t[["mentions", "neg_rate", "priority"]].round(3))

print(f"\nANSWER: Northerly should address the app or website first "
      f"({top2.neg_rate['app-website']:.1%} negative rate across "
      f"{top2.mentions['app-website']} mentions), followed by phone support "
      f"({top2.neg_rate['phone']:.1%} across {top2.mentions['phone']}).")

T8-H1  |  dataset: hard  |  answerable: yes
Which 2 issues should Northerly address first? Consider both negative rate and volume equally, and ignore any topics with fewer than 30 mentions.
----------------------------------------------------------------------------------------------------
                       mentions  neg_rate  priority
child_aspect                                       
app-website                 192     0.328     0.744
phone                        40     0.650     0.531
attitude-of-staff            42     0.548     0.456
speed                       139     0.151     0.440
ease-of-use                 144     0.021     0.352
price-value-for-money        30     0.200     0.142
discounts-promotions         35     0.171     0.135

ANSWER: Northerly should address the app or website first (32.8% negative rate across 192 mentions), followed by phone support (65.0% across 40).


In [143]:
df = ask("T8-H2")

d = df[df.org == "CompareHive"]
topics = (d.groupby("child_aspect")
            .agg(mentions=("sentiment", "size"),
                 negatives=("sentiment", lambda s: (s == "negative").sum()))
            .assign(neg_rate=lambda x: x.negatives / x.mentions)
            .sort_values("neg_rate", ascending=False))

t = topics.query("mentions >= @MIN_N and child_aspect not in @NON_ACTIONABLE").copy()

mm = lambda s: (s - s.min()) / (s.max() - s.min())
t["rate_scaled"] = mm(t.neg_rate)
t["vol_scaled"] = mm(t.mentions)
t["priority"] = 0.5 * t.rate_scaled + 0.5 * t.vol_scaled

t = t.sort_values("priority", ascending=False)
top2 = t.head(2)
print(t[["mentions", "neg_rate", "priority"]].round(3))

print(f"\nANSWER: CompareHive should address the app or website first "
      f"({top2.neg_rate['app-website']:.1%} negative rate across "
      f"{top2.mentions['app-website']} mentions), followed by price and value for money "
      f"({top2.neg_rate['price-value-for-money']:.1%} across "
      f"{top2.mentions['price-value-for-money']}). ")

T8-H2  |  dataset: hard  |  answerable: yes
Which 2 issues should CompareHive address first? Consider both negative rate and volume equally, and ignore any topics with fewer than 30 mentions.
----------------------------------------------------------------------------------------------------
                       mentions  neg_rate  priority
child_aspect                                       
app-website                  80     0.575     0.678
price-value-for-money       158     0.127     0.572
ease-of-use                 147     0.163     0.561
attitude-of-staff           103     0.107     0.326
phone                        37     0.351     0.286
speed                        39     0.051     0.008

ANSWER: CompareHive should address the app or website first (57.5% negative rate across 80 mentions), followed by price and value for money (12.7% across 158). 


In [144]:
df = ask("T8-H3")

d = df[df.org == "Quantly"]
topics = (d.groupby("child_aspect")
            .agg(mentions=("sentiment", "size"),
                 negatives=("sentiment", lambda s: (s == "negative").sum()))
            .assign(neg_rate=lambda x: x.negatives / x.mentions)
            .sort_values("neg_rate", ascending=False))

t = topics.query("mentions >= @MIN_N and child_aspect not in @NON_ACTIONABLE").copy()

mm = lambda s: (s - s.min()) / (s.max() - s.min())
t["rate_scaled"] = mm(t.neg_rate)
t["vol_scaled"] = mm(t.mentions)
t["priority"] = 0.5 * t.rate_scaled + 0.5 * t.vol_scaled

t = t.sort_values("priority", ascending=False)
top2 = t.head(2)
print(t[["mentions", "neg_rate", "priority"]].round(3))

print(f"\nANSWER: Quantly should address the app or website first "
      f"({top2.neg_rate['app-website']:.1%} negative rate across "
      f"{top2.mentions['app-website']} mentions), followed by staff attitude "
      f"({top2.neg_rate['attitude-of-staff']:.1%} across {top2.mentions['attitude-of-staff']}). ")

T8-H3  |  dataset: hard  |  answerable: yes
Which 2 issues should Quantly address first? Consider both negative rate and volume equally, and ignore any topics with fewer than 30 mentions.
----------------------------------------------------------------------------------------------------
                   mentions  neg_rate  priority
child_aspect                                   
app-website             227     0.159     0.639
attitude-of-staff        54     0.315     0.500
ease-of-use             112     0.098     0.168

ANSWER: Quantly should address the app or website first (15.9% negative rate across 227 mentions), followed by staff attitude (31.5% across 54). 


## T9 - Comparative diagnosis and recommendation

In [60]:
df = ask("T9-E1")

focus, peer = "Halden Savings", "Kestrel Bank"

# 1. overall complaint rate = negative reviews / all reviews, per organisation
overall = (df[df.org.isin([focus, peer])]
             .assign(is_neg=lambda d: d.sentiment.eq("negative"))
             .groupby("org")
             .agg(reviews=("sentiment", "size"), complaints=("is_neg", "sum"))
             .reindex([focus, peer]))
overall["complaint_rate"] = (overall.complaints / overall.reviews).round(4)
print("overall complaint rate:"); print(overall)
gap = overall.complaint_rate[focus] - overall.complaint_rate[peer]
print(f"\ngap = {gap * 100:.1f} pp higher at {focus}\n")

# 2. per-topic comparison, plus each topic's contribution to the overall gap:
#    contribution = mix-weighted negative rate at focus - the same at peer, which sums to the gap
rows = []
for topic in sorted(set(df[df.org.isin([focus, peer])].child_aspect)):
    r = {"topic": topic}
    for tag, org in (("f", focus), ("p", peer)):
        s = df[(df.org == org) & (df.child_aspect == topic)]
        r[f"n_{tag}"] = len(s)
        r[f"rate_{tag}"] = s.sentiment.eq("negative").mean() if len(s) else np.nan
        r[f"w_{tag}"] = len(s) / (df.org == org).sum()
    r["contribution_pp"] = 100 * (np.nan_to_num(r["w_f"] * r["rate_f"])
                                  - np.nan_to_num(r["w_p"] * r["rate_p"]))
    rows.append(r)
cmp = (pd.DataFrame(rows).set_index("topic")
         .assign(rate_diff_pp=lambda d: ((d.rate_f - d.rate_p) * 100).round(1))
         .round(4).sort_values("contribution_pp", ascending=False))
print(f"per-topic ({focus} = _f, {peer} = _p):")
print(cmp[["n_f", "rate_f", "n_p", "rate_p", "rate_diff_pp", "contribution_pp"]])
print(f"\ncontributions sum to {cmp.contribution_pp.sum():.1f} pp = the overall gap\n")

# 3. the recommendation pool: the diagnosis above covers every topic, but the
#    non-actionable topics are not something the organisation can act on
fixable = cmp[~cmp.index.isin(NON_ACTIONABLE)]
print(f"recommendation pool, excluding {NON_ACTIONABLE}:")
print(fixable[["n_f", "rate_f", "n_p", "rate_p", "rate_diff_pp", "contribution_pp"]])
biggest, pick = cmp.index[0], fixable.index[0]
print(f"\nlargest contributor = {biggest}, largest actionable contributor = {pick}")

# 4. two-proportion z-tests on every topic both organisations cover with >= MIN_N mentions
shared = cmp[(cmp.n_f >= MIN_N) & (cmp.n_p >= MIN_N)].copy()
pvals = []
for topic, r in shared.iterrows():
    k1, n1 = round(r.rate_f * r.n_f), int(r.n_f)
    k2, n2 = round(r.rate_p * r.n_p), int(r.n_p)
    p_pool = (k1 + k2) / (n1 + n2)
    z = (k1 / n1 - k2 / n2) / np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
    pvals.append(2 * stats.norm.sf(abs(z)))
shared["p_value"] = np.round(pvals, 4)
print(f"\nshared topics (both >= {MIN_N} mentions), tested:")
print(shared[["n_f", "rate_f", "n_p", "rate_p", "rate_diff_pp", "p_value"]]
      .sort_values("rate_diff_pp", key=abs))

print(f"\nANSWER: {focus} should address account access first. The largest contributor to its "
      f"higher complaint rate ({overall.complaint_rate[focus]:.1%} versus "
      f"{overall.complaint_rate[peer]:.1%} for {peer}, a {gap * 100:.1f} pp gap) is account "
      f"access, at an {cmp.rate_f['account-access']:.1%} negative rate "
      f"(n={cmp.n_f['account-access']}), compared to {cmp.rate_p['account-access']:.1%} "
      f"(n={cmp.n_p['account-access']}) for {peer}. The "
      f"{abs(cmp.rate_diff_pp['account-access']):.1f} pp gap on account access is statistically "
      f"significant (p = {shared.p_value['account-access']:.3f}). Both companies are comparable "
      f"on email support ({cmp.rate_f['email']:.1%} vs {cmp.rate_p['email']:.1%}) and ease of use "
      f"({cmp.rate_f['ease-of-use']:.1%} vs {cmp.rate_p['ease-of-use']:.1%}), and neither "
      f"difference is significant.")

T9-E1  |  dataset: easy  |  answerable: yes
Why does Halden Savings have a higher complaint rate than Kestrel Bank? What should Halden Savings improve first?
----------------------------------------------------------------------------------------------------
overall complaint rate:
                reviews  complaints  complaint_rate
org                                                
Halden Savings      571         284          0.4974
Kestrel Bank        709         276          0.3893

gap = 10.8 pp higher at Halden Savings



per-topic (Halden Savings = _f, Kestrel Bank = _p):


                       n_f  rate_f  n_p  rate_p  rate_diff_pp  contribution_pp
topic                                                                         
account-access          40  0.8000   40  0.5500          25.0           2.5012
discounts-promotions    40  0.8250   40  0.6000          22.5           2.3943
app-website             62  0.3871  139  0.1223          26.5           1.8054
speed                   40  0.2750   40  0.0250          25.0           1.7854
general-satisfaction    85  0.3765  110  0.2818           9.5           1.2318
email                   40  0.8250   40  0.8250           0.0           1.1249
attitude-of-staff       40  0.8750   40  0.9500          -7.5           0.7699
competitor              40  0.2000   40  0.1250           7.5           0.6958
phone                   40  0.6500   40  0.8250         -17.5          -0.1010
price-value-for-money   40  0.3000   40  0.4000         -10.0          -0.1551
reviews                 40  0.6000   40  0.8000     

In [61]:
df = ask("T9-E2")

focus, peer = "CompareHive", "PricePilot"

overall = (df[df.org.isin([focus, peer])]
             .assign(is_neg=lambda d: d.sentiment.eq("negative"))
             .groupby("org")
             .agg(reviews=("sentiment", "size"), complaints=("is_neg", "sum"))
             .reindex([focus, peer]))
overall["complaint_rate"] = (overall.complaints / overall.reviews).round(4)
print("overall complaint rate:"); print(overall)
gap = overall.complaint_rate[focus] - overall.complaint_rate[peer]
print(f"\ngap = {gap * 100:.1f} pp higher at {focus}\n")

rows = []
for topic in sorted(set(df[df.org.isin([focus, peer])].child_aspect)):
    r = {"topic": topic}
    for tag, org in (("f", focus), ("p", peer)):
        s = df[(df.org == org) & (df.child_aspect == topic)]
        r[f"n_{tag}"] = len(s)
        r[f"rate_{tag}"] = s.sentiment.eq("negative").mean() if len(s) else np.nan
        r[f"w_{tag}"] = len(s) / (df.org == org).sum()
    r["contribution_pp"] = 100 * (np.nan_to_num(r["w_f"] * r["rate_f"])
                                  - np.nan_to_num(r["w_p"] * r["rate_p"]))
    rows.append(r)
cmp = (pd.DataFrame(rows).set_index("topic")
         .assign(rate_diff_pp=lambda d: ((d.rate_f - d.rate_p) * 100).round(1))
         .round(4).sort_values("contribution_pp", ascending=False))
print(f"per-topic ({focus} = _f, {peer} = _p):")
print(cmp[["n_f", "rate_f", "n_p", "rate_p", "rate_diff_pp", "contribution_pp"]])
print(f"\ncontributions sum to {cmp.contribution_pp.sum():.1f} pp = the overall gap\n")

fixable = cmp[~cmp.index.isin(NON_ACTIONABLE)]
print(f"recommendation pool, excluding {NON_ACTIONABLE}:")
print(fixable[["n_f", "rate_f", "n_p", "rate_p", "rate_diff_pp", "contribution_pp"]])
biggest, pick = cmp.index[0], fixable.index[0]
print(f"\nlargest contributor = {biggest}, largest actionable contributor = {pick}")

shared = cmp[(cmp.n_f >= MIN_N) & (cmp.n_p >= MIN_N)].copy()
pvals = []
for topic, r in shared.iterrows():
    k1, n1 = round(r.rate_f * r.n_f), int(r.n_f)
    k2, n2 = round(r.rate_p * r.n_p), int(r.n_p)
    p_pool = (k1 + k2) / (n1 + n2)
    z = (k1 / n1 - k2 / n2) / np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
    pvals.append(2 * stats.norm.sf(abs(z)))
shared["p_value"] = np.round(pvals, 4)
print(f"\nshared topics (both >= {MIN_N} mentions), tested:")
print(shared[["n_f", "rate_f", "n_p", "rate_p", "rate_diff_pp", "p_value"]]
      .sort_values("rate_diff_pp", key=abs))

print(f"\nANSWER: {focus} should address the app or website first. The largest contributor to its "
      f"higher complaint rate ({overall.complaint_rate[focus]:.1%} versus "
      f"{overall.complaint_rate[peer]:.1%} for {peer}, a {gap * 100:.1f} pp gap) is general "
      f"satisfaction, at a {cmp.rate_f['general-satisfaction']:.1%} negative rate "
      f"(n={cmp.n_f['general-satisfaction']}), compared to "
      f"{cmp.rate_p['general-satisfaction']:.1%} (n={cmp.n_p['general-satisfaction']}) for {peer}. "
      f"General satisfaction is not something {focus} can act on directly, so the first fix is the "
      f"app or website, at a {cmp.rate_f['app-website']:.1%} negative rate "
      f"(n={cmp.n_f['app-website']}), compared to {cmp.rate_p['app-website']:.1%} "
      f"(n={cmp.n_p['app-website']}) for {peer}. The "
      f"{abs(cmp.rate_diff_pp['app-website']):.1f} pp gap on the app or website is statistically "
      f"significant (p < 0.001). Both companies are comparable on phone support "
      f"({cmp.rate_f['phone']:.1%} vs {cmp.rate_p['phone']:.1%}) and staff attitude "
      f"({cmp.rate_f['attitude-of-staff']:.1%} vs {cmp.rate_p['attitude-of-staff']:.1%}), and "
      f"neither difference is significant.")

T9-E2  |  dataset: easy  |  answerable: yes
Why does CompareHive have a higher complaint rate than PricePilot? What should CompareHive improve first?
----------------------------------------------------------------------------------------------------


overall complaint rate:


             reviews  complaints  complaint_rate
org                                             
CompareHive      986         325          0.3296
PricePilot       927         191          0.2060

gap = 12.4 pp higher at CompareHive



per-topic (CompareHive = _f, PricePilot = _p):
                       n_f  rate_f  n_p  rate_p  rate_diff_pp  contribution_pp
topic                                                                         
general-satisfaction   190  0.3053  122  0.0820          22.3           4.8036
app-website            100  0.7400   79  0.3418          39.8           4.5924
ease-of-use            147  0.1633   89  0.0787           8.5           1.6790
discounts-promotions    40  0.7250   40  0.4000          32.5           1.2152
reviews                 40  0.6500   40  0.3750          27.5           1.0188
price-value-for-money  135  0.1556  159  0.0818           7.4           0.7274
competitor              54  0.2778   51  0.1569          12.1           0.6583
phone                   40  0.4750   40  0.4750           0.0          -0.1226
attitude-of-staff      103  0.1068  175  0.0800           2.7          -0.3946
account-access          40  0.8750   40  0.9250          -5.0          -0.4417
speed

In [62]:
df = ask("T9-H1")

focus, peer = "Sable Row", "Northerly"

overall = (df[df.org.isin([focus, peer])]
             .assign(is_neg=lambda d: d.sentiment.eq("negative"))
             .groupby("org")
             .agg(reviews=("sentiment", "size"), complaints=("is_neg", "sum"))
             .reindex([focus, peer]))
overall["complaint_rate"] = (overall.complaints / overall.reviews).round(4)
print("overall complaint rate:"); print(overall)
gap = overall.complaint_rate[focus] - overall.complaint_rate[peer]
print(f"\ngap = {gap * 100:.1f} pp higher at {focus}\n")

rows = []
for topic in sorted(set(df[df.org.isin([focus, peer])].child_aspect)):
    r = {"topic": topic}
    for tag, org in (("f", focus), ("p", peer)):
        s = df[(df.org == org) & (df.child_aspect == topic)]
        r[f"n_{tag}"] = len(s)
        r[f"rate_{tag}"] = s.sentiment.eq("negative").mean() if len(s) else np.nan
        r[f"w_{tag}"] = len(s) / (df.org == org).sum()
    r["contribution_pp"] = 100 * (np.nan_to_num(r["w_f"] * r["rate_f"])
                                  - np.nan_to_num(r["w_p"] * r["rate_p"]))
    rows.append(r)
cmp = (pd.DataFrame(rows).set_index("topic")
         .assign(rate_diff_pp=lambda d: ((d.rate_f - d.rate_p) * 100).round(1))
         .round(4).sort_values("contribution_pp", ascending=False))
print(f"per-topic ({focus} = _f, {peer} = _p):")
print(cmp[["n_f", "rate_f", "n_p", "rate_p", "rate_diff_pp", "contribution_pp"]])
print(f"\ncontributions sum to {cmp.contribution_pp.sum():.1f} pp = the overall gap\n")

fixable = cmp[~cmp.index.isin(NON_ACTIONABLE)]
print(f"recommendation pool, excluding {NON_ACTIONABLE}:")
print(fixable[["n_f", "rate_f", "n_p", "rate_p", "rate_diff_pp", "contribution_pp"]])
biggest, pick = cmp.index[0], fixable.index[0]
print(f"\nlargest contributor = {biggest}, largest actionable contributor = {pick}")

shared = cmp[(cmp.n_f >= MIN_N) & (cmp.n_p >= MIN_N)].copy()
pvals = []
for topic, r in shared.iterrows():
    k1, n1 = round(r.rate_f * r.n_f), int(r.n_f)
    k2, n2 = round(r.rate_p * r.n_p), int(r.n_p)
    p_pool = (k1 + k2) / (n1 + n2)
    z = (k1 / n1 - k2 / n2) / np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
    pvals.append(2 * stats.norm.sf(abs(z)))
shared["p_value"] = np.round(pvals, 4)
print(f"\nshared topics (both >= {MIN_N} mentions), tested:")
print(shared[["n_f", "rate_f", "n_p", "rate_p", "rate_diff_pp", "p_value"]]
      .sort_values("rate_diff_pp", key=abs))

print(f"\nANSWER: {focus} should address staff attitude first. The largest contributor to its "
      f"higher complaint rate ({overall.complaint_rate[focus]:.1%} versus "
      f"{overall.complaint_rate[peer]:.1%} for {peer}, a {gap * 100:.1f} pp gap) is general "
      f"satisfaction, at a {cmp.rate_f['general-satisfaction']:.1%} negative rate "
      f"(n={cmp.n_f['general-satisfaction']}), compared to "
      f"{cmp.rate_p['general-satisfaction']:.1%} (n={cmp.n_p['general-satisfaction']}) for {peer}. "
      f"General satisfaction is not something {focus} can act on directly, so the first fix is "
      f"staff attitude, at a {cmp.rate_f['attitude-of-staff']:.1%} negative rate "
      f"(n={cmp.n_f['attitude-of-staff']}), compared to {cmp.rate_p['attitude-of-staff']:.1%} "
      f"(n={cmp.n_p['attitude-of-staff']}) for {peer}. The "
      f"{abs(cmp.rate_diff_pp['attitude-of-staff']):.1f} pp gap on staff attitude is "
      f"statistically significant (p = {shared.p_value['attitude-of-staff']:.3f}). The only topic "
      f"the two are comparable on is price and value for money "
      f"({cmp.rate_f['price-value-for-money']:.1%} vs {cmp.rate_p['price-value-for-money']:.1%}), "
      f"where the difference is not significant.")

T9-H1  |  dataset: hard  |  answerable: yes
Why does Sable Row have a higher complaint rate than Northerly? What should Sable Row improve first?
----------------------------------------------------------------------------------------------------
overall complaint rate:
           reviews  complaints  complaint_rate
org                                           
Sable Row      925         279          0.3016
Northerly      876         173          0.1975

gap = 10.4 pp higher at Sable Row



per-topic (Sable Row = _f, Northerly = _p):
                       n_f  rate_f  n_p  rate_p  rate_diff_pp  contribution_pp
topic                                                                         
general-satisfaction   354  0.3079  184  0.0272          28.1          11.2130
attitude-of-staff       56  0.7857   42  0.5476          23.8           2.1312
ease-of-use             74  0.2838  144  0.0208          26.3           1.9278
speed                  100  0.3000  139  0.1511          14.9           0.8460
price-value-for-money   40  0.2000   30  0.2000           0.0           0.1799
email                    8  0.8750   10  0.7000          17.5          -0.0423
reviews                  5  1.0000    6  1.0000           0.0          -0.1444
discounts-promotions    29  0.1724   35  0.1714           0.1          -0.1444
competitor               0     NaN   46  0.0435           NaN          -0.2283
account-access           0     NaN    8  0.6250           NaN          -0.5708
app-webs


shared topics (both >= 30 mentions), tested:


                       n_f  rate_f  n_p  rate_p  rate_diff_pp  p_value
topic                                                                 
price-value-for-money   40  0.2000   30  0.2000           0.0   1.0000
app-website            253  0.1818  192  0.3281         -14.6   0.0004
speed                  100  0.3000  139  0.1511          14.9   0.0056
attitude-of-staff       56  0.7857   42  0.5476          23.8   0.0121
ease-of-use             74  0.2838  144  0.0208          26.3   0.0000
general-satisfaction   354  0.3079  184  0.0272          28.1   0.0000

ANSWER: Sable Row should address staff attitude first. The largest contributor to its higher complaint rate (30.2% versus 19.8% for Northerly, a 10.4 pp gap) is general satisfaction, at a 30.8% negative rate (n=354), compared to 2.7% (n=184) for Northerly. General satisfaction is not something Sable Row can act on directly, so the first fix is staff attitude, at a 78.6% negative rate (n=56), compared to 54.8% (n=42) for Norther

In [63]:
df = ask("T9-H2")

focus, peer = "Larkmead Market", "Oakpan Grocers"

overall = (df[df.org.isin([focus, peer])]
             .assign(is_neg=lambda d: d.sentiment.eq("negative"))
             .groupby("org")
             .agg(reviews=("sentiment", "size"), complaints=("is_neg", "sum"))
             .reindex([focus, peer]))
overall["complaint_rate"] = (overall.complaints / overall.reviews).round(4)
print("overall complaint rate:"); print(overall)
gap = overall.complaint_rate[focus] - overall.complaint_rate[peer]
print(f"\ngap = {gap * 100:.1f} pp higher at {focus}\n")

rows = []
for topic in sorted(set(df[df.org.isin([focus, peer])].child_aspect)):
    r = {"topic": topic}
    for tag, org in (("f", focus), ("p", peer)):
        s = df[(df.org == org) & (df.child_aspect == topic)]
        r[f"n_{tag}"] = len(s)
        r[f"rate_{tag}"] = s.sentiment.eq("negative").mean() if len(s) else np.nan
        r[f"w_{tag}"] = len(s) / (df.org == org).sum()
    r["contribution_pp"] = 100 * (np.nan_to_num(r["w_f"] * r["rate_f"])
                                  - np.nan_to_num(r["w_p"] * r["rate_p"]))
    rows.append(r)
cmp = (pd.DataFrame(rows).set_index("topic")
         .assign(rate_diff_pp=lambda d: ((d.rate_f - d.rate_p) * 100).round(1))
         .round(4).sort_values("contribution_pp", ascending=False))
print(f"per-topic ({focus} = _f, {peer} = _p):")
print(cmp[["n_f", "rate_f", "n_p", "rate_p", "rate_diff_pp", "contribution_pp"]])
print(f"\ncontributions sum to {cmp.contribution_pp.sum():.1f} pp = the overall gap\n")

fixable = cmp[~cmp.index.isin(NON_ACTIONABLE)]
print(f"recommendation pool, excluding {NON_ACTIONABLE}:")
print(fixable[["n_f", "rate_f", "n_p", "rate_p", "rate_diff_pp", "contribution_pp"]])
biggest, pick = cmp.index[0], fixable.index[0]
print(f"\nlargest contributor = {biggest}, largest actionable contributor = {pick}")

shared = cmp[(cmp.n_f >= MIN_N) & (cmp.n_p >= MIN_N)].copy()
pvals = []
for topic, r in shared.iterrows():
    k1, n1 = round(r.rate_f * r.n_f), int(r.n_f)
    k2, n2 = round(r.rate_p * r.n_p), int(r.n_p)
    p_pool = (k1 + k2) / (n1 + n2)
    z = (k1 / n1 - k2 / n2) / np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
    pvals.append(2 * stats.norm.sf(abs(z)))
shared["p_value"] = np.round(pvals, 4)
print(f"\nshared topics (both >= {MIN_N} mentions), tested:")
print(shared[["n_f", "rate_f", "n_p", "rate_p", "rate_diff_pp", "p_value"]]
      .sort_values("rate_diff_pp", key=abs))

# the peer has no mentions of the app or website, so that gap cannot be tested
print(f"\nANSWER: {focus} should address the app or website first. The largest contributor to its "
      f"higher complaint rate ({overall.complaint_rate[focus]:.1%} versus "
      f"{overall.complaint_rate[peer]:.1%} for {peer}, a {gap * 100:.1f} pp gap) is the app or "
      f"website, at a {cmp.rate_f['app-website']:.1%} negative rate "
      f"(n={cmp.n_f['app-website']}), compared to no mentions for {peer}. Both companies are "
      f"comparable on staff attitude ({cmp.rate_f['attitude-of-staff']:.1%} vs "
      f"{cmp.rate_p['attitude-of-staff']:.1%}) and discounts and promotions "
      f"({cmp.rate_f['discounts-promotions']:.1%} vs {cmp.rate_p['discounts-promotions']:.1%}), "
      f"and neither difference is significant.")

T9-H2  |  dataset: hard  |  answerable: yes
Why does Larkmead Market have a higher complaint rate than Oakpan Grocers? What should Larkmead Market improve first?
----------------------------------------------------------------------------------------------------
overall complaint rate:
                 reviews  complaints  complaint_rate
org                                                 
Larkmead Market      658         293          0.4453
Oakpan Grocers       397         124          0.3123

gap = 13.3 pp higher at Larkmead Market



per-topic (Larkmead Market = _f, Oakpan Grocers = _p):
                       n_f  rate_f  n_p  rate_p  rate_diff_pp  contribution_pp
topic                                                                         
app-website            189  0.4603    0     NaN           NaN          13.2219
general-satisfaction    96  0.3438   56  0.1071          23.7           3.5039
speed                   51  0.5686   38  0.2368          33.2           2.1403
discounts-promotions    54  0.4074   44  0.2500          15.7           0.5727
phone                    3  0.6667    0     NaN           NaN           0.3040
email                    1  1.0000    1  1.0000           0.0          -0.0999
price-value-for-money   21  0.0476   13  0.2308         -18.3          -0.6037
attitude-of-staff       40  0.1750   38  0.2368          -6.2          -1.2032
account-access          16  0.6250   18  0.6111           1.4          -1.2510
competitor              17  0.1765   25  0.2800         -10.4          -1.30

In [64]:
df = ask("T9-H3")

focus, peer = "Halden Savings", "Northeast Bank"

overall = (df[df.org.isin([focus, peer])]
             .assign(is_neg=lambda d: d.sentiment.eq("negative"))
             .groupby("org")
             .agg(reviews=("sentiment", "size"), complaints=("is_neg", "sum"))
             .reindex([focus, peer]))
overall["complaint_rate"] = (overall.complaints / overall.reviews).round(4)
print("overall complaint rate:"); print(overall)
gap = overall.complaint_rate[focus] - overall.complaint_rate[peer]
print(f"\ngap = {gap * 100:.1f} pp higher at {focus}\n")

rows = []
for topic in sorted(set(df[df.org.isin([focus, peer])].child_aspect)):
    r = {"topic": topic}
    for tag, org in (("f", focus), ("p", peer)):
        s = df[(df.org == org) & (df.child_aspect == topic)]
        r[f"n_{tag}"] = len(s)
        r[f"rate_{tag}"] = s.sentiment.eq("negative").mean() if len(s) else np.nan
        r[f"w_{tag}"] = len(s) / (df.org == org).sum()
    r["contribution_pp"] = 100 * (np.nan_to_num(r["w_f"] * r["rate_f"])
                                  - np.nan_to_num(r["w_p"] * r["rate_p"]))
    rows.append(r)
cmp = (pd.DataFrame(rows).set_index("topic")
         .assign(rate_diff_pp=lambda d: ((d.rate_f - d.rate_p) * 100).round(1))
         .round(4).sort_values("contribution_pp", ascending=False))
print(f"per-topic ({focus} = _f, {peer} = _p):")
print(cmp[["n_f", "rate_f", "n_p", "rate_p", "rate_diff_pp", "contribution_pp"]])
print(f"\ncontributions sum to {cmp.contribution_pp.sum():.1f} pp = the overall gap\n")

fixable = cmp[~cmp.index.isin(NON_ACTIONABLE)]
print(f"recommendation pool, excluding {NON_ACTIONABLE}:")
print(fixable[["n_f", "rate_f", "n_p", "rate_p", "rate_diff_pp", "contribution_pp"]])
biggest, pick = cmp.index[0], fixable.index[0]
print(f"\nlargest contributor = {biggest}, largest actionable contributor = {pick}")

shared = cmp[(cmp.n_f >= MIN_N) & (cmp.n_p >= MIN_N)].copy()
pvals = []
for topic, r in shared.iterrows():
    k1, n1 = round(r.rate_f * r.n_f), int(r.n_f)
    k2, n2 = round(r.rate_p * r.n_p), int(r.n_p)
    p_pool = (k1 + k2) / (n1 + n2)
    z = (k1 / n1 - k2 / n2) / np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
    pvals.append(2 * stats.norm.sf(abs(z)))
shared["p_value"] = np.round(pvals, 4)
print(f"\nshared topics (both >= {MIN_N} mentions), tested:")
print(shared[["n_f", "rate_f", "n_p", "rate_p", "rate_diff_pp", "p_value"]]
      .sort_values("rate_diff_pp", key=abs))

# the peer has no mentions of staff attitude, so that gap cannot be tested
print(f"\nANSWER: {focus} should address staff attitude first. The largest contributor to its "
      f"higher complaint rate ({overall.complaint_rate[focus]:.1%} versus "
      f"{overall.complaint_rate[peer]:.1%} for {peer}, a {gap * 100:.1f} pp gap) is staff "
      f"attitude, at a {cmp.rate_f['attitude-of-staff']:.1%} negative rate "
      f"(n={cmp.n_f['attitude-of-staff']}), compared to no mentions for {peer}. Both companies "
      f"are comparable on ease of use ({cmp.rate_f['ease-of-use']:.1%} vs "
      f"{cmp.rate_p['ease-of-use']:.1%}) and speed ({cmp.rate_f['speed']:.1%} vs "
      f"{cmp.rate_p['speed']:.1%}), and neither difference is significant.")

T9-H3  |  dataset: hard  |  answerable: yes
Why does Halden Savings have a higher complaint rate than Northeast Bank? What should Halden Savings improve first?
----------------------------------------------------------------------------------------------------
overall complaint rate:
                reviews  complaints  complaint_rate
org                                                
Halden Savings      336         125           0.372
Northeast Bank      417          83           0.199

gap = 17.3 pp higher at Halden Savings



per-topic (Halden Savings = _f, Northeast Bank = _p):
                       n_f  rate_f  n_p  rate_p  rate_diff_pp  contribution_pp
topic                                                                         
attitude-of-staff       22  0.9545    0     NaN           NaN           6.2500
general-satisfaction    80  0.3875  103  0.1359          25.2           5.8689
price-value-for-money   29  0.5517   34  0.0000          55.2           4.7619
account-access          27  0.5926    0     NaN           NaN           4.7619
ease-of-use            107  0.2617   90  0.3333          -7.2           1.1391
discounts-promotions     2  1.0000    1  1.0000           0.0           0.3554
email                    2  1.0000    1  1.0000           0.0           0.3554
reviews                  1  1.0000    1  1.0000           0.0           0.0578
phone                    0     NaN    1  1.0000           NaN          -0.2398
competitor              26  0.0769   33  0.1515          -7.5          -0.603


shared topics (both >= 30 mentions), tested:
                      n_f  rate_f  n_p  rate_p  rate_diff_pp  p_value
topic                                                                
ease-of-use           107  0.2617   90  0.3333          -7.2   0.2717
speed                  40  0.1500   40  0.2500         -10.0   0.2636
general-satisfaction   80  0.3875  103  0.1359          25.2   0.0001

ANSWER: Halden Savings should address staff attitude first. The largest contributor to its higher complaint rate (37.2% versus 19.9% for Northeast Bank, a 17.3 pp gap) is staff attitude, at a 95.5% negative rate (n=22), compared to no mentions for Northeast Bank. Both companies are comparable on ease of use (26.2% vs 33.3%) and speed (15.0% vs 25.0%), and neither difference is significant.


## T10 - Full CX report

In [65]:
df = ask("T10-E1")

org, industry = "Trippa", "Travel Booking"
d = df[df.org == org]
ind = df[df.industry == industry]

# 1. overall sentiment
mix = d.sentiment.value_counts().to_frame("reviews")
mix["share"] = mix.reviews / len(d)
print(f"1. OVERALL SENTIMENT - {org} ({industry}), {len(d)} reviews"); print(mix)

# 2. count by topic, compute the negative rate, drop the topics below the mention floor
topics = (d.groupby("child_aspect")
            .agg(mentions=("sentiment", "size"),
                 negatives=("sentiment", lambda s: (s == "negative").sum()))
            .assign(neg_rate=lambda x: x.negatives / x.mentions)
            .sort_values("neg_rate", ascending=False))
kept = topics.query("mentions >= @MIN_N")
print(f"\n2. PAIN POINTS ranked by negative rate (>= {MIN_N} mentions)"); print(kept)
print("\n   (all topics, for reference)"); print(topics)

# 3. top issue vs the industry average, two-sided two-proportion z-test with pooled variance
top = kept.index[0]
ind_top = ind[ind.child_aspect == top]
k1, n1 = kept.negatives[top], kept.mentions[top]
k2, n2 = (ind_top.sentiment == "negative").sum(), len(ind_top)
org_rate, ind_rate = k1 / n1, k2 / n2
p_pool = (k1 + k2) / (n1 + n2)
z = (org_rate - ind_rate) / np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
p = 2 * stats.norm.sf(abs(z))
print(f"\n3. TOP ISSUE vs INDUSTRY - account access")
print(f"   {org}: {org_rate:.1%} negative (n = {n1})")
print(f"   {industry} average: {ind_rate:.1%} negative (n = {n2})")
print(f"   gap = {(org_rate - ind_rate) * 100:+.1f} pp   (z = {z:.3f}, p = {p:.4f})")

# 4. prioritised roadmap: drop the non-actionable topics, then score with the same
#    Priority = 0.5 * minmax(negative rate) + 0.5 * minmax(volume) as T8, and take the top 3
road = kept.query("child_aspect not in @NON_ACTIONABLE").copy()
mm = lambda s: (s - s.min()) / (s.max() - s.min())
road["rate_scaled"] = mm(road.neg_rate)
road["vol_scaled"] = mm(road.mentions)
road["priority"] = 0.5 * road.rate_scaled + 0.5 * road.vol_scaled
road["industry_rate"] = [(ind[ind.child_aspect == tp].sentiment == "negative").mean()
                         for tp in road.index]
road = road.sort_values("priority", ascending=False)
top3 = road.head(3)
print(f"\n4. PRIORITISED ROADMAP (equal weight on negative rate and volume,"
      f" excluding {NON_ACTIONABLE})"); print(road)
print("\ntop 3:"); print(top3)

# roadmap verbs: rank 1 is the priority pick, so it is always "Improve"; ranks 2 and 3
# take their verb from the industry comparison, so "Maintain" means already better than average
print(f"""
ANSWER - CX report for {org}
  Overview: {org} received {len(d)} labelled mentions - {mix.share['positive']:.1%} positive, \
{mix.share['negative']:.1%} negative, {mix.share['neutral']:.1%} neutral. Addressing the app or \
website first is likely to be most impactful.
  Top pain points:
    1. account access ({kept.neg_rate['account-access']:.1%} negative, \
{kept.mentions['account-access']} mentions)
    2. email support ({kept.neg_rate['email']:.1%} negative, {kept.mentions['email']} mentions)
    3. phone support ({kept.neg_rate['phone']:.1%} negative, {kept.mentions['phone']} mentions)
  Industry comparison: {org}'s complaint rate about account access ({org_rate:.1%}) is lower than \
the {industry} industry average of {ind_rate:.1%}, but the difference is not statistically \
significant (p = {p:.3f}).
  Roadmap:
    1. Improve the app or website ({top3.neg_rate['app-website']:.1%} negative across \
{top3.mentions['app-website']} mentions, below the {industry} average of \
{top3.industry_rate['app-website']:.1%})
    2. Maintain account access ({top3.neg_rate['account-access']:.1%} negative across \
{top3.mentions['account-access']} mentions, below the {industry} average of \
{top3.industry_rate['account-access']:.1%})
    3. Improve email support ({top3.neg_rate['email']:.1%} negative across \
{top3.mentions['email']} mentions, above the {industry} average of \
{top3.industry_rate['email']:.1%})""")

T10-E1  |  dataset: easy  |  answerable: yes
Write a CX report for Trippa. Cover overall sentiment, top pain points ranked by negative rate, and how the top issue compares with the industry average. Also give me a prioritised improvement roadmap.
----------------------------------------------------------------------------------------------------
1. OVERALL SENTIMENT - Trippa (Travel Booking), 674 reviews
           reviews   share
sentiment                 
positive       379  0.5623
negative       285  0.4228
neutral         10  0.0148

2. PAIN POINTS ranked by negative rate (>= 30 mentions)
                       mentions  negatives  neg_rate
child_aspect                                        
account-access               40         35    0.8750
email                        40         35    0.8750
phone                        40         26    0.6500
attitude-of-staff            40         24    0.6000
discounts-promotions         40         23    0.5750
general-satisfaction        1

In [66]:
df = ask("T10-E2")

org, industry = "Kestrel Bank", "Banking"
d = df[df.org == org]
ind = df[df.industry == industry]

mix = d.sentiment.value_counts().to_frame("reviews")
mix["share"] = mix.reviews / len(d)
print(f"1. OVERALL SENTIMENT - {org} ({industry}), {len(d)} reviews"); print(mix)

topics = (d.groupby("child_aspect")
            .agg(mentions=("sentiment", "size"),
                 negatives=("sentiment", lambda s: (s == "negative").sum()))
            .assign(neg_rate=lambda x: x.negatives / x.mentions)
            .sort_values("neg_rate", ascending=False))
kept = topics.query("mentions >= @MIN_N")
print(f"\n2. PAIN POINTS ranked by negative rate (>= {MIN_N} mentions)"); print(kept)
print("\n   (all topics, for reference)"); print(topics)

top = kept.index[0]
ind_top = ind[ind.child_aspect == top]
k1, n1 = kept.negatives[top], kept.mentions[top]
k2, n2 = (ind_top.sentiment == "negative").sum(), len(ind_top)
org_rate, ind_rate = k1 / n1, k2 / n2
p_pool = (k1 + k2) / (n1 + n2)
z = (org_rate - ind_rate) / np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
p = 2 * stats.norm.sf(abs(z))
print(f"\n3. TOP ISSUE vs INDUSTRY - staff attitude")
print(f"   {org}: {org_rate:.1%} negative (n = {n1})")
print(f"   {industry} average: {ind_rate:.1%} negative (n = {n2})")
print(f"   gap = {(org_rate - ind_rate) * 100:+.1f} pp   (z = {z:.3f}, p = {p:.4f})")

# 4. prioritised roadmap: drop the non-actionable topics, then score with the same
#    Priority = 0.5 * minmax(negative rate) + 0.5 * minmax(volume) as T8, and take the top 3
road = kept.query("child_aspect not in @NON_ACTIONABLE").copy()
mm = lambda s: (s - s.min()) / (s.max() - s.min())
road["rate_scaled"] = mm(road.neg_rate)
road["vol_scaled"] = mm(road.mentions)
road["priority"] = 0.5 * road.rate_scaled + 0.5 * road.vol_scaled
road["industry_rate"] = [(ind[ind.child_aspect == tp].sentiment == "negative").mean()
                         for tp in road.index]
road = road.sort_values("priority", ascending=False)
top3 = road.head(3)
print(f"\n4. PRIORITISED ROADMAP (equal weight on negative rate and volume,"
      f" excluding {NON_ACTIONABLE})"); print(road)
print("\ntop 3:"); print(top3)

print(f"""
ANSWER - CX report for {org}
  Overview: {org} received {len(d)} labelled mentions - {mix.share['positive']:.1%} positive, \
{mix.share['negative']:.1%} negative, {mix.share['neutral']:.1%} neutral. Addressing the app or \
website first is likely to be most impactful.
  Top pain points:
    1. staff attitude ({kept.neg_rate['attitude-of-staff']:.1%} negative, \
{kept.mentions['attitude-of-staff']} mentions)
    2. email support ({kept.neg_rate['email']:.1%} negative, {kept.mentions['email']} mentions)
    3. phone support ({kept.neg_rate['phone']:.1%} negative, {kept.mentions['phone']} mentions)
  Industry comparison: {org}'s complaint rate about staff attitude ({org_rate:.1%}) is higher than \
the {industry} industry average of {ind_rate:.1%}, but the difference is not statistically \
significant (p = {p:.3f}).
  Roadmap:
    1. Improve the app or website ({top3.neg_rate['app-website']:.1%} negative across \
{top3.mentions['app-website']} mentions, below the {industry} average of \
{top3.industry_rate['app-website']:.1%})
    2. Improve staff attitude ({top3.neg_rate['attitude-of-staff']:.1%} negative across \
{top3.mentions['attitude-of-staff']} mentions, above the {industry} average of \
{top3.industry_rate['attitude-of-staff']:.1%})
    3. Improve email support ({top3.neg_rate['email']:.1%} negative across \
{top3.mentions['email']} mentions, above the {industry} average of \
{top3.industry_rate['email']:.1%})""")

T10-E2  |  dataset: easy  |  answerable: yes
Write a CX report for Kestrel Bank. Cover overall sentiment, top pain points ranked by negative rate, and how the top issue compares with the industry average. Also give me a prioritised improvement roadmap.
----------------------------------------------------------------------------------------------------
1. OVERALL SENTIMENT - Kestrel Bank (Banking), 709 reviews
           reviews   share
sentiment                 
positive       365  0.5148
negative       276  0.3893
neutral         68  0.0959

2. PAIN POINTS ranked by negative rate (>= 30 mentions)
                       mentions  negatives  neg_rate
child_aspect                                        
attitude-of-staff            40         38    0.9500
email                        40         33    0.8250
phone                        40         33    0.8250
reviews                      40         32    0.8000
discounts-promotions         40         24    0.6000
account-access          


4. PRIORITISED ROADMAP (equal weight on negative rate and volume, excluding ['competitor', 'general-satisfaction', 'reviews'])
                       mentions  negatives  neg_rate  rate_scaled  vol_scaled  priority  industry_rate
child_aspect                                                                                          
app-website                 139         17    0.1223       0.1052      1.0000    0.5526         0.1874
attitude-of-staff            40         38    0.9500       1.0000      0.0000    0.5000         0.9000
email                        40         33    0.8250       0.8649      0.0000    0.4324         0.8125
phone                        40         33    0.8250       0.8649      0.0000    0.4324         0.6500
ease-of-use                 100         24    0.2400       0.2324      0.6061    0.4192         0.2562
discounts-promotions         40         24    0.6000       0.6216      0.0000    0.3108         0.7875
account-access               40         22    0.

In [67]:
df = ask("T10-H1")

org, industry = "Vella & Co", "Fashion"
d = df[df.org == org]
ind = df[df.industry == industry]

mix = d.sentiment.value_counts().to_frame("reviews")
mix["share"] = mix.reviews / len(d)
print(f"1. OVERALL SENTIMENT - {org} ({industry}), {len(d)} reviews"); print(mix)

topics = (d.groupby("child_aspect")
            .agg(mentions=("sentiment", "size"),
                 negatives=("sentiment", lambda s: (s == "negative").sum()))
            .assign(neg_rate=lambda x: x.negatives / x.mentions)
            .sort_values("neg_rate", ascending=False))
kept = topics.query("mentions >= @MIN_N")
print(f"\n2. PAIN POINTS ranked by negative rate (>= {MIN_N} mentions)"); print(kept)
print("\n   (all topics, for reference)"); print(topics)

top = kept.index[0]
ind_top = ind[ind.child_aspect == top]
k1, n1 = kept.negatives[top], kept.mentions[top]
k2, n2 = (ind_top.sentiment == "negative").sum(), len(ind_top)
org_rate, ind_rate = k1 / n1, k2 / n2
p_pool = (k1 + k2) / (n1 + n2)
z = (org_rate - ind_rate) / np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
p = 2 * stats.norm.sf(abs(z))
print(f"\n3. TOP ISSUE vs INDUSTRY - phone support")
print(f"   {org}: {org_rate:.1%} negative (n = {n1})")
print(f"   {industry} average: {ind_rate:.1%} negative (n = {n2})")
print(f"   gap = {(org_rate - ind_rate) * 100:+.1f} pp   (z = {z:.3f}, p = {p:.4f})")

# 4. prioritised roadmap: drop the non-actionable topics, then score with the same
#    Priority = 0.5 * minmax(negative rate) + 0.5 * minmax(volume) as T8, and take the top 3
road = kept.query("child_aspect not in @NON_ACTIONABLE").copy()
mm = lambda s: (s - s.min()) / (s.max() - s.min())
road["rate_scaled"] = mm(road.neg_rate)
road["vol_scaled"] = mm(road.mentions)
road["priority"] = 0.5 * road.rate_scaled + 0.5 * road.vol_scaled
road["industry_rate"] = [(ind[ind.child_aspect == tp].sentiment == "negative").mean()
                         for tp in road.index]
road = road.sort_values("priority", ascending=False)
top3 = road.head(3)
print(f"\n4. PRIORITISED ROADMAP (equal weight on negative rate and volume,"
      f" excluding {NON_ACTIONABLE})"); print(road)
print("\ntop 3:"); print(top3)

print(f"""
ANSWER - CX report for {org}
  Overview: {org} received {len(d)} labelled mentions - {mix.share['positive']:.1%} positive, \
{mix.share['negative']:.1%} negative, {mix.share['neutral']:.1%} neutral. Addressing speed first \
is likely to be most impactful.
  Top pain points:
    1. phone support ({kept.neg_rate['phone']:.1%} negative, {kept.mentions['phone']} mentions)
    2. staff attitude ({kept.neg_rate['attitude-of-staff']:.1%} negative, \
{kept.mentions['attitude-of-staff']} mentions)
    3. price and value for money ({kept.neg_rate['price-value-for-money']:.1%} negative, \
{kept.mentions['price-value-for-money']} mentions)
  Industry comparison: {org}'s complaint rate about phone support ({org_rate:.1%}) is higher than \
the {industry} industry average of {ind_rate:.1%}, but the difference is not statistically \
significant (p = {p:.3f}).
  Roadmap:
    1. Improve speed ({top3.neg_rate['speed']:.1%} negative across {top3.mentions['speed']} \
mentions, above the {industry} average of {top3.industry_rate['speed']:.1%})
    2. Improve phone support ({top3.neg_rate['phone']:.1%} negative across \
{top3.mentions['phone']} mentions, above the {industry} average of \
{top3.industry_rate['phone']:.1%})
    3. Maintain the app or website ({top3.neg_rate['app-website']:.1%} negative across \
{top3.mentions['app-website']} mentions, below the {industry} average of \
{top3.industry_rate['app-website']:.1%})""")

T10-H1  |  dataset: hard  |  answerable: yes
Write a CX report for Vella & Co. Cover overall sentiment, top pain points ranked by negative rate, and how the top issue compares with the industry average. Also give me a prioritised improvement roadmap.
----------------------------------------------------------------------------------------------------
1. OVERALL SENTIMENT - Vella & Co (Fashion), 777 reviews
           reviews   share
sentiment                 
positive       579  0.7452
negative       189  0.2432
neutral          9  0.0116

2. PAIN POINTS ranked by negative rate (>= 30 mentions)
                       mentions  negatives  neg_rate
child_aspect                                        
phone                        40         32    0.8000
attitude-of-staff            58         28    0.4828
price-value-for-money        36         15    0.4167
speed                       150         44    0.2933
app-website                 156         29    0.1859
competitor                  


4. PRIORITISED ROADMAP (equal weight on negative rate and volume, excluding ['competitor', 'general-satisfaction', 'reviews'])
                       mentions  negatives  neg_rate  rate_scaled  vol_scaled  priority  industry_rate
child_aspect                                                                                          
speed                       150         44    0.2933       0.1749      0.9500    0.5625         0.2131
phone                        40         32    0.8000       1.0000      0.0333    0.5167         0.7447
app-website                 156         29    0.1859       0.0000      1.0000    0.5000         0.2748
attitude-of-staff            58         28    0.4828       0.4834      0.1833    0.3334         0.5498
price-value-for-money        36         15    0.4167       0.3758      0.0000    0.1879         0.3245

top 3:
              mentions  negatives  neg_rate  rate_scaled  vol_scaled  priority  industry_rate
child_aspect                                     

In [68]:
df = ask("T10-H2")

org, industry = "Freshbury", "Groceries"
d = df[df.org == org]
ind = df[df.industry == industry]

mix = d.sentiment.value_counts().to_frame("reviews")
mix["share"] = mix.reviews / len(d)
print(f"1. OVERALL SENTIMENT - {org} ({industry}), {len(d)} reviews"); print(mix)

topics = (d.groupby("child_aspect")
            .agg(mentions=("sentiment", "size"),
                 negatives=("sentiment", lambda s: (s == "negative").sum()))
            .assign(neg_rate=lambda x: x.negatives / x.mentions)
            .sort_values("neg_rate", ascending=False))
kept = topics.query("mentions >= @MIN_N")
print(f"\n2. PAIN POINTS ranked by negative rate (>= {MIN_N} mentions)"); print(kept)
print("\n   (all topics, for reference)"); print(topics)

top = kept.index[0]
ind_top = ind[ind.child_aspect == top]
k1, n1 = kept.negatives[top], kept.mentions[top]
k2, n2 = (ind_top.sentiment == "negative").sum(), len(ind_top)
org_rate, ind_rate = k1 / n1, k2 / n2
p_pool = (k1 + k2) / (n1 + n2)
z = (org_rate - ind_rate) / np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
p = 2 * stats.norm.sf(abs(z))
print(f"\n3. TOP ISSUE vs INDUSTRY - the app or website")
print(f"   {org}: {org_rate:.1%} negative (n = {n1})")
print(f"   {industry} average: {ind_rate:.1%} negative (n = {n2})")
print(f"   gap = {(org_rate - ind_rate) * 100:+.1f} pp   (z = {z:.3f}, p = {p:.4f})")

# 4. prioritised roadmap: drop the non-actionable topics, then score with the same
#    Priority = 0.5 * minmax(negative rate) + 0.5 * minmax(volume) as T8, and take the top 3
road = kept.query("child_aspect not in @NON_ACTIONABLE").copy()
mm = lambda s: (s - s.min()) / (s.max() - s.min())
road["rate_scaled"] = mm(road.neg_rate)
road["vol_scaled"] = mm(road.mentions)
road["priority"] = 0.5 * road.rate_scaled + 0.5 * road.vol_scaled
road["industry_rate"] = [(ind[ind.child_aspect == tp].sentiment == "negative").mean()
                         for tp in road.index]
road = road.sort_values("priority", ascending=False)
top3 = road.head(3)
print(f"\n4. PRIORITISED ROADMAP (equal weight on negative rate and volume,"
      f" excluding {NON_ACTIONABLE})"); print(road)
print("\ntop 3:"); print(top3)

print(f"""
ANSWER - CX report for {org}
  Overview: {org} received {len(d)} labelled mentions - {mix.share['positive']:.1%} positive, \
{mix.share['negative']:.1%} negative, {mix.share['neutral']:.1%} neutral. Addressing the app or \
website first is likely to be most impactful.
  Top pain points:
    1. the app or website ({kept.neg_rate['app-website']:.1%} negative, \
{kept.mentions['app-website']} mentions)
    2. speed ({kept.neg_rate['speed']:.1%} negative, {kept.mentions['speed']} mentions)
    3. ease of use ({kept.neg_rate['ease-of-use']:.1%} negative, \
{kept.mentions['ease-of-use']} mentions)
  Industry comparison: {org}'s complaint rate about the app or website ({org_rate:.1%}) is higher \
than the {industry} industry average of {ind_rate:.1%}, but the difference is not statistically \
significant (p = {p:.3f}).
  Roadmap:
    1. Improve the app or website ({top3.neg_rate['app-website']:.1%} negative across \
{top3.mentions['app-website']} mentions, above the {industry} average of \
{top3.industry_rate['app-website']:.1%})
    2. Maintain ease of use ({top3.neg_rate['ease-of-use']:.1%} negative across \
{top3.mentions['ease-of-use']} mentions, below the {industry} average of \
{top3.industry_rate['ease-of-use']:.1%})
    3. Maintain speed ({top3.neg_rate['speed']:.1%} negative across {top3.mentions['speed']} \
mentions, below the {industry} average of {top3.industry_rate['speed']:.1%})""")

T10-H2  |  dataset: hard  |  answerable: yes
Write a CX report for Freshbury. Cover overall sentiment, top pain points ranked by negative rate, and how the top issue compares with the industry average. Also give me a prioritised improvement roadmap.
----------------------------------------------------------------------------------------------------


1. OVERALL SENTIMENT - Freshbury (Groceries), 677 reviews
           reviews   share
sentiment                 
positive       401  0.5923
negative       263  0.3885
neutral         13  0.0192

2. PAIN POINTS ranked by negative rate (>= 30 mentions)
                      mentions  negatives  neg_rate
child_aspect                                       
app-website                262        130    0.4962
speed                       54         21    0.3889
ease-of-use                189         57    0.3016
attitude-of-staff           40         12    0.3000
general-satisfaction        73         20    0.2740

   (all topics, for reference)
                       mentions  negatives  neg_rate
child_aspect                                        
reviews                       3          3    1.0000
phone                         2          2    1.0000
email                         1          1    1.0000
competitor                   19         10    0.5263
app-website                 262     

In [69]:
# abstention: only one topic clears the mention floor
df = ask("T10-H3")

org = "Lumora"
d = df[df.org == org]
industry = d.industry.iloc[0]

topics = (d.groupby("child_aspect")
            .agg(mentions=("sentiment", "size"),
                 negatives=("sentiment", lambda s: (s == "negative").sum()))
            .assign(neg_rate=lambda x: x.negatives / x.mentions)
            .sort_values("neg_rate", ascending=False))
print(f"{org} ({industry}) reviews = {len(d)}, topics covered = {topics.shape[0]} of 12")
print("sentiment mix:"); print(d.sentiment.value_counts().to_string())
print("\ntopics:"); print(topics)

kept = topics.query("mentions >= @MIN_N")
print(f"\ntopics clearing the {MIN_N}-mention floor = {len(kept)}: {list(kept.index)}")
print(f"\n{industry} industry coverage in the dataset:")
print(df[df.industry == industry].child_aspect.value_counts().to_string())

print(f"\nANSWER: No answer. {org} has {len(d)} reviews spread over {topics.shape[0]} of the 12 "
      f"topics, and only one")
print(f"of them (account access, {kept.mentions['account-access']} mentions) clears the "
      f"{MIN_N}-mention floor. A single eligible")
print("topic cannot be ranked into 'top pain points' and cannot be turned into a prioritised")
print("roadmap, so the report as requested cannot be produced. Only the overall sentiment")
print("split is supportable, which is one component of four.")

T10-H3  |  dataset: hard  |  answerable: no (abstain)
Write a CX report for Lumora. Cover overall sentiment, top pain points ranked by negative rate, and how the top issue compares with the industry average. Also give me a prioritised improvement roadmap.
----------------------------------------------------------------------------------------------------
Lumora (Streaming) reviews = 56, topics covered = 3 of 12
sentiment mix:
sentiment
neutral     32
negative    18
positive     6

topics:
                      mentions  negatives  neg_rate
child_aspect                                       
account-access              40         14    0.3500
app-website                 12          4    0.3333
discounts-promotions         4          0    0.0000

topics clearing the 30-mention floor = 1: ['account-access']

Streaming industry coverage in the dataset:
child_aspect
account-access          89
app-website             46
discounts-promotions    11

ANSWER: No answer. Lumora has 56 reviews spr

# End of notebook